# 🏀 Category-Dominance Draft Engine — 2026-27
## Swiss See Ballers · ESPN H2H Each-Category Salary-Cap Auction

**Draft: Oct 17, 2026 @ 20:00 · 14 teams · $200 cap · 13 roster spots · 10 categories**

---

## The thesis, and the arithmetic that decides it

In **Head-to-Head Each Category** you win a matchup by taking **6 of 10 categories**.
Nothing else in the box score matters. Winning PTS by 400 pays exactly what winning it by
1 pays, and a category lost 0.499–0.501 costs exactly what one lost by a mile costs.

So the objective is **not** "build the best team." It is:

$$\max \; P\big(\text{win} \ge 6 \text{ of } 10 \text{ categories}\big)$$

That change of objective is the entire strategy, and it has four consequences this notebook
is built around.

### 1. Balance is a losing strategy — measurably, not stylistically

A balanced roster sits near the league mean in all 10 categories. Near the mean means every
category is near a coin flip, so your matchup is 10 coin flips and you need 6 heads — and
against rivals who are *also* near the mean, you play pure variance every week.
**Cell 11 measures this directly: concentrating is worth roughly +19 percentage points of
matchup win probability over the balanced build, spending the same $200.** In a 14-team league
with 4 playoff spots, that gap is the difference between contending and coin-flipping.

### 2. A dollar spent on a category you lose anyway is a wasted dollar

This is why punting works, and it is an argument about *budget*, not basketball. Concede a
category completely and you never pay for it again. Your whole $200 goes into the categories
that decide matchups, while a balanced rival spends a large share of his on production that
is irrelevant to the head-to-head. **You are effectively playing at a ~1.6× budget in the
categories that count.**

### 3. What matters is how many categories are *in play*, not how many you target

Start with the arithmetic. Target exactly 6 categories, each a 90% lock, and you must win **all
six**:

$$P = 0.90^6 = 0.53$$

A coin flip. If 7 are in play at that strength you need only 6 of 7:

$$P = 0.90^7 + 7 \cdot 0.90^6 \cdot 0.10 = 0.85$$

**Same per-category strength, 0.53 → 0.85, from holding a single spare.** So the quantity to
maximise is *categories in play*, and six targets with nothing behind them is the version that
loses to variance.

The catch: that calculation assumes strength stays at 90% as you add targets, and your budget does
not allow it — spreading $200 over more categories weakens all of them. So the target count is a
real trade-off between buffer and dilution, it is **not settled by algebra**, and Cell 12 settles
it empirically every time you run it.

Be warned that the answer moves. Across revisions of this notebook the winning plan has come back
as 5, 6, 7 and 8 targets, flipping whenever the inputs got more accurate — when retired players
were removed, and again when real ESPN market prices replaced the model's. The top plans routinely
sit within 0.05 of each other. **Read Cell 12's live table; do not trust a count written here.**

What *is* stable, across every version: **the winning plan always ends up with more categories in
play than it pays for.** Targets bought for one reason drag others into range for free. A measured
profile from one run:

| | categories | weekly P(win) | paid for? |
|---|---|---|---|
| **Locked** | REB · AST · DD | 0.90 – 0.96 | yes |
| **Leaning** | PTS · STL | 0.63 – 0.67 | PTS yes; **STL free** |
| **Live coin-flips** | FG% · FT% · BLK · PPM | 0.44 – 0.54 | FG%/FT%/BLK **free**; PPM paid, not paying |
| **Conceded** | 3PM | 0.29 | no |

Five targets bought; **nine categories in play**, six of them at even money or better, needing
six. The spare capacity was not purchased — it arrived with the tall, rebounding, playmaking
players the plan wanted anyway. Note the wart the notebook flags on its own: **PPM is a target
that is not paying** (0.44), riding along with PTS rather than earning its own dollars, so never
bid *extra* for a points-per-minute specialist.

> **So the brief holds.** "Find categories you can dominate and always win" is exactly right, and
> the data is emphatic: concentrating beats balance by **+17 to +23 percentage points**, and the
> balanced build measures **0.52** — a literal coin flip. The only correction is to the count:
> aim to have **seven or more categories in play**, however many you actually pay for, because six
> locks with no spare is a 0.53 proposition. Choose targets for their correlation and the spare
> capacity comes free.

### 4. Which categories — measured, never assumed

Two properties make a category cheap to dominate, and both are measurable:

| Property | Why it makes the category cheap | Cell |
|---|---|---|
| **Correlation** with your other targets | One purchase pays into several categories at once — PTS and PPM correlate at **0.90** here, so points nearly buys points-per-minute for free | 7 |
| **Concentration** (top-heaviness) | If 20 players own most of the league's supply, 2–3 buys win it outright | 7 |

Cell 11 then brute-forces **all 628 category bundles** of size 5–10, builds an optimal
roster for each under real ESPN roster rules, simulates weekly totals against a 14-team
field, and ranks by P(win matchup). **The punt plan is an output of this notebook, not an
input to it** — and the answer it returns is not the obvious one.

---

## Five things done differently here

Each of these changes *which players you buy*, not just the bookkeeping.

**1. Weekly totals, scored the way ESPN actually scores them.** A category is decided by the
**10 starters'** aggregate over the week: counting stats (PTS/REB/AST/STL/BLK/3PM/DD) are
*summed over games played*; ratio stats are *team aggregates* (FG% = ΣFGM/ΣFGA,
PPM = ΣPTS/ΣMIN). Ranking rosters by the mean of per-player per-game averages — the
intuitive move — silently counts your bench, ignores that production accumulates by games
played, and turns ratio categories into an average of averages. Cell 9 does it properly.

**2. Ratio categories are volume-weighted.** A 92% free-throw shooter taking one attempt a
game is worth almost nothing to your team FT%. What moves a team ratio is
$(\text{player rate} - \text{league rate}) \times \text{volume}$. Z-scoring the raw
percentage ranks low-volume specialists as elite; Cell 8 fixes it, and it re-orders the
FG%/FT% boards substantially.

**3. Probability, not point estimates.** "Always win" is a statement about variance, so every
category is scored as **P(win this week)**, simulated from each player's real game-to-game
standard deviation. A category won by a 2% margin flips half the time; one won by 30% never
does. Point estimates cannot tell those apart, and they are entirely different assets.

**4. Market price and your price are different numbers.** The auction clears at what the
*league* will pay, and that side is taken from **ESPN's own published auction values** (Cell 5B)
rather than guessed at by a model. Your value is strategy-conditional.
**The gap is the whole edge:** Cell 13 ranks by `SURPLUS = my value in $ − market price`.
Pricing your own board and bidding those numbers is self-defeating; it makes you pay full
retail for precisely the players your strategy already told you to want.

**5. A real auction, not a draft queue.** Cell 15 runs a simultaneous **English auction**:
14 budgeted agents, rotating nominations, each bidding to a private ceiling, players clearing
at **second-highest willingness-to-pay + $1**, with inflation tracked as money and talent
leave the room. That is what produces per-team, per-player prices — and it is *why* bargains
exist: a player costs only what the **second**-most-interested manager will pay.

---

### Cell map

| # | Cell |
|---|---|
| 1–2 | Imports · **League rules (ESPN — single source of truth)** |
| 3–5 | Game logs · player metadata · 2026-27 rosters (all disk-cached) |
| 5B | **Real ESPN market data** — auction values + true position eligibility |
| 6 | Projections **with per-game variance** |
| 7 | **Category structure**: correlation + concentration → which cats are cheap |
| 8 | Valuation: volume-weighted ratios, market price vs strategy value |
| 9 | **ESPN weekly scoring engine** (starters, sums, team ratios) |
| 10 | Roster optimiser (budget · 10 ESPN slots · max 4 C) |
| 11 | **Bundle search — all 628 category plans ranked by P(win matchup)** |
| 12 | Robustness: what happens when rivals contest your categories |
| 13 | Your team + the surplus board (who to actually buy) |
| 14 | **Full auction simulation — every team, every player, every price** |
| 15 | Season Monte Carlo: 18 matchups, record, playoff odds |
| 16 | Draft-day cheat sheet · Excel export |
| 17 | Export to the live draft assistant (`draft_app/`) |

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# CELL 1 — IMPORTS
# ═══════════════════════════════════════════════════════════════════
import os, time, math, itertools, warnings
import pandas as pd
import numpy as np
warnings.filterwarnings('ignore')

from nba_api.stats.endpoints import leaguegamelog, commonplayerinfo, commonteamroster
from nba_api.stats.static import teams as static_teams

pd.set_option('display.width', 220)
pd.set_option('display.max_columns', 60)

CACHE_DIR = 'cache'
os.makedirs(CACHE_DIR, exist_ok=True)
RNG_SEED = 20261017          # the draft date as master seed -> every result reproducible

print('✅ Imports OK  ·  pandas', pd.__version__, '· numpy', np.__version__)

---
## Cell 2 · League rules — the single source of truth

Every ESPN setting lives in this one cell; nothing downstream hardcodes a rule. Two settings
deserve comment because they change *which players are worth buying*, not just bookkeeping.

**`MAX_CENTERS = 4`** turns out to be the binding constraint on the strategy the search
lands on. Only Center is capped by ESPN — every other slot is unlimited. Since the winning
plan is built on rebounds, double-doubles and blocks, this cap is what stops it from
degenerating into "buy nine centers," and the way around it is **PF-eligible rebounders**,
who supply the same categories without consuming a Center slot. That is visible in Cell 13's
roster, and it is a real draft-day instruction.

**`DURABILITY_EXPONENT` is derived, not tuned.** Three settings compound to make availability
worth more here than a plain games-played haircut implies:

- `IR_SLOTS = 0` — an injured player still occupies 1 of your 13 spots. There is nowhere to park him.
- `LINEUP_PROTECTION = False` — an injured starter is *not* swapped out for you. He posts zeros.
- `LINEUP_CHANGES = WEEKLY` — lineups lock at each player's first game, so a Monday injury cannot be undone. You eat the whole matchup.

So value scales as `availability ** DURABILITY_EXPONENT` — convex, not linear. In a league
with an IR slot, lineup protection and daily changes the exponent reverts to 1.0 on its own.
**This matters double for a category-lock strategy: a lock is only a lock if the player is on
the floor.** A 90%-certain category with an absent star is a coin flip, which is precisely
the failure mode the whole plan is built to avoid.

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# CELL 2 — LEAGUE RULES  (single source of truth)
# ═══════════════════════════════════════════════════════════════════
# Transcribed from the ESPN settings page:
# fantasy.espn.com/basketball/league/settings?leagueId=1015252784&seasonId=2027
# If ESPN changes a setting, this cell is the ONLY place that needs editing.

LEAGUE_NAME    = 'Swiss See Ballers'
YOUR_TEAM_NAME = 'Janartuo Ninjas'
TEAM_NAMES = [
    'Nordstern Hawks', 'Brick City', 'Transalpes Trail Shredders',
    'Primitive Analytics', 'Janartuo Ninjas', 'Inaugural Champs',
    'Ulifortada', 'St. Wendelin Storm', 'Switchler to the moon',
    'Mobiliar Mullets', 'Load Management Slyrix', 'Undersized Problems',
    'Team 15', "Noah's Nasty Team",
]
NUM_TEAMS    = len(TEAM_NAMES)          # 14
SCORING_TYPE = 'H2H_EACH_CATEGORY'

# ── Categories: ESPN label -> internal key ───────────────────────────────────
# This league scores DD and PPM and does NOT score turnovers, so these 10 are the complete
# category set. PPM (points per minute) is rare in ESPN leagues and turns out to matter.
CATEGORY_MAP = {
    'FG%': 'FG_PCT', 'FT%': 'FT_PCT', '3PM': 'FG3M',  'REB': 'REB', 'AST': 'AST',
    'STL': 'STL',    'BLK': 'BLK',    'DD':  'DD',    'PTS': 'PTS', 'PPM': 'PTS_MIN',
}
CATS = list(CATEGORY_MAP.values())
LBL  = {v: k for k, v in CATEGORY_MAP.items()}          # 'FG3M' -> '3PM'
KEY  = dict(CATEGORY_MAP)                               # '3PM'  -> 'FG3M'
CATS_TO_WIN = 6                                         # 6 of 10 wins the matchup

# How each category is settled in a weekly matchup:
COUNTING_CATS = ['PTS', 'REB', 'AST', 'STL', 'BLK', 'FG3M', 'DD']   # summed over games
RATIO_CATS    = ['FG_PCT', 'FT_PCT', 'PTS_MIN']                     # team aggregates
assert set(COUNTING_CATS) | set(RATIO_CATS) == set(CATS)

# ── Draft ────────────────────────────────────────────────────────────────────
DRAFT_TYPE      = 'SALARY_CAP'
DRAFT_DATE      = pd.Timestamp('2026-10-17 20:00')
BUDGET_PER_TEAM = 200
MIN_BID         = 1
TOTAL_BUDGET    = NUM_TEAMS * BUDGET_PER_TEAM           # $2,800

# ── Roster ───────────────────────────────────────────────────────────────────
ROSTER_SIZE    = 13
BENCH_COUNT    = 3
IR_SLOTS       = 0
TOTAL_DRAFTED  = NUM_TEAMS * ROSTER_SIZE                # 182
ALL_POSITIONS  = ['PG', 'SG', 'SF', 'PF', 'C']
STARTER_SLOTS  = [
    ('PG', ['PG']), ('SG', ['SG']), ('SF', ['SF']), ('PF', ['PF']), ('C', ['C']),
    ('G',  ['PG', 'SG']), ('F', ['SF', 'PF']),
    ('UTIL', ALL_POSITIONS), ('UTIL', ALL_POSITIONS), ('UTIL', ALL_POSITIONS),
]
TOTAL_STARTERS    = len(STARTER_SLOTS)                  # 10
SLOT_MASKS        = [frozenset(a) for _, a in STARTER_SLOTS]
POSITION_MAXIMUMS = {'C': 4}                            # only Center is capped
MAX_CENTERS       = POSITION_MAXIMUMS['C']
assert ROSTER_SIZE == TOTAL_STARTERS + BENCH_COUNT + IR_SLOTS

# ── Lineups / acquisitions ───────────────────────────────────────────────────
LINEUP_CHANGES    = 'WEEKLY_LOCK_AT_FIRST_GAME'
LINEUP_PROTECTION = False
FAB_BUDGET        = 100
MATCHUP_ACQUISITION_LIMIT = 7

# ── Season ───────────────────────────────────────────────────────────────────
WEEKS_PER_MATCHUP       = 1
REGULAR_SEASON_MATCHUPS = 18
PLAYOFF_TEAMS           = 4
CATEGORY_TIE_BREAKER    = None      # "No tie breakers" -> an exact tie stays a tie
KEEPERS                 = False     # pure redraft: no future value in any price
GAMES_PER_WEEK          = 3.45      # NBA average team games in a 7-day matchup

# ── Rule-derived durability penalty (see markdown above) ────────────────────
DURABILITY_EXPONENT = 1.0
if IR_SLOTS == 0:                        DURABILITY_EXPONENT += 0.20
if not LINEUP_PROTECTION:                DURABILITY_EXPONENT += 0.10
if LINEUP_CHANGES.startswith('WEEKLY'):  DURABILITY_EXPONENT += 0.05
DURABILITY_EXPONENT = round(DURABILITY_EXPONENT, 2)

print(f'📋 {LEAGUE_NAME}: {NUM_TEAMS} teams · {SCORING_TYPE} · ${BUDGET_PER_TEAM} cap')
print(f'   Roster {ROSTER_SIZE} = {TOTAL_STARTERS} starters + {BENCH_COUNT} bench + {IR_SLOTS} IR'
      f'   |   max {MAX_CENTERS} C   |   {len(CATS)} cats, need {CATS_TO_WIN} to win')
print(f'   Counting cats (summed over games): {", ".join(LBL[c] for c in COUNTING_CATS)}')
print(f'   Ratio cats (team aggregate)      : {", ".join(LBL[c] for c in RATIO_CATS)}')
print(f'   Durability exponent (from IR=0 + no protection + weekly lock): {DURABILITY_EXPONENT}')
print(f'   Draft: {DRAFT_DATE:%b %d, %Y @ %H:%M}')

---
## Cells 3–5 · Data

Two seasons of game logs (2025-26 weighted 70%, 2024-25 weighted 30%), player metadata for
age/height/position, and the **current 2026-27 rosters** so the teammate-context adjustment
knows where players actually play now, after the summer's trades and free agency.

Everything is **cached to disk on first run**. `stats.nba.com` throttles aggressive clients,
and re-fetching 53,000 game-log rows on every kernel restart is how you get rate-limited on
draft day. Delete the files in `cache/` to force a refresh.

Why only two seasons: recent form predicts next season better than decade-old data, and `GP`
within these two seasons is the availability signal the durability term needs.

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# CELL 3 — GAME LOGS (disk-cached)
# ═══════════════════════════════════════════════════════════════════
SEASONS       = ['2024-25', '2025-26']
CURRENT_SEASON, PREV_SEASON = '2025-26', '2024-25'
W_CURRENT, W_PREV = 0.70, 0.30      # recency weighting of the two seasons
MAX_GAMES     = 82

def fetch_gamelogs(seasons):
    frames = []
    for s in seasons:
        f = f'{CACHE_DIR}/gamelogs_{s}.csv'
        if os.path.exists(f):
            d = pd.read_csv(f)
            print(f'  📦 {s}: {len(d):,} rows from cache')
        else:
            print(f'  🌐 {s}: fetching from stats.nba.com ...', end='', flush=True)
            d = leaguegamelog.LeagueGameLog(
                season=s, player_or_team_abbreviation='P',
                season_type_all_star='Regular Season').get_data_frames()[0]
            d.to_csv(f, index=False)
            print(f' {len(d):,} rows -> cached')
            time.sleep(2.5)
        d['SEASON'] = s
        frames.append(d)
    return pd.concat(frames, ignore_index=True)

def parse_minutes(v):
    if isinstance(v, str) and ':' in v:
        a, b = v.split(':'); return int(a) + int(b) / 60
    try:    return float(v) if pd.notna(v) else 0.0
    except: return 0.0

df_logs = fetch_gamelogs(SEASONS)
df_logs['MIN']       = df_logs['MIN'].apply(parse_minutes)
df_logs['GAME_DATE'] = pd.to_datetime(df_logs['GAME_DATE'])
# A double-double = 10+ in two of PTS/REB/AST/STL/BLK, evaluated per game.
df_logs['DD'] = ((df_logs[['PTS','REB','AST','STL','BLK']] >= 10).sum(axis=1) >= 2).astype(int)

print(f"\n✅ {len(df_logs):,} player-games  ·  {df_logs['PLAYER_ID'].nunique():,} players")
print(df_logs.groupby('SEASON').agg(games=('GAME_ID','nunique'), rows=('PTS','size')).to_string())

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# CELL 4 — PLAYER METADATA: age · height · position (disk-cached)
# ═══════════════════════════════════════════════════════════════════
# Only players who appeared in 2025-26 are fetched (~400, not every player since 2015).
# Chunked with pauses and checkpointed after every chunk, so an interrupted run resumes
# instead of starting over.
#
# KNOWN LIMITATION: NBA.com exposes only coarse positions (Guard / Forward / Center and
# combos), never ESPN's PG vs SG vs SF vs PF. expand_position() below approximates ESPN
# eligibility from the coarse label plus height. It is a heuristic -- good enough to keep the
# roster optimiser honest about needing bigs vs guards, but CHECK ESPN for any player you are
# counting on to fill one specific starter slot.

META_FILE = f'{CACHE_DIR}/player_meta_{CURRENT_SEASON}.csv'
CHUNK, CHUNK_PAUSE, CALL_PAUSE = 25, 3.0, 0.6

need_ids = sorted(df_logs.loc[df_logs.SEASON == CURRENT_SEASON, 'PLAYER_ID'].unique())
have = pd.read_csv(META_FILE) if os.path.exists(META_FILE) else pd.DataFrame(columns=['PLAYER_ID'])
todo = [p for p in need_ids if p not in set(have['PLAYER_ID'])]
print(f'metadata: {len(have)} cached, {len(todo)} to fetch')

rows = []
for start in range(0, len(todo), CHUNK):
    for pid in todo[start:start+CHUNK]:
        for attempt in (1, 2):
            try:
                info = commonplayerinfo.CommonPlayerInfo(player_id=pid).get_data_frames()[0]
                h = info['HEIGHT'].iloc[0]
                ft_in = [int(x) for x in h.split('-')] if isinstance(h, str) and '-' in h else None
                rows.append({'PLAYER_ID': pid,
                             'BIRTH_DATE': info['BIRTHDATE'].iloc[0],
                             'POSITION':   info['POSITION'].iloc[0],
                             'HEIGHT_IN':  ft_in[0]*12 + ft_in[1] if ft_in else np.nan,
                             'DRAFT_YEAR': info['DRAFT_YEAR'].iloc[0],
                             'COUNTRY':    info['COUNTRY'].iloc[0]})
                time.sleep(CALL_PAUSE); break
            except Exception as e:
                if attempt == 2: print(f'  warn {pid}: {e}')
                else: time.sleep(2.0)
    if rows:
        have = pd.concat([have, pd.DataFrame(rows)], ignore_index=True); rows = []
        have.to_csv(META_FILE, index=False)
        print(f'  checkpoint: {len(have)} players cached')
    time.sleep(CHUNK_PAUSE)

df_meta = have.drop_duplicates('PLAYER_ID').copy()

def expand_position(position, height_in):
    # Coarse NBA.com label + height -> the set of ESPN slots a player plausibly fills.
    p = (position or '').strip()
    h = height_in if pd.notna(height_in) else 78
    if   p == 'Guard':          s = {'PG','SG'} if h <= 76 else {'SG'}
    elif p == 'Guard-Forward':  s = {'SG','SF'} | ({'PG'} if h <= 76 else set())
    elif p == 'Forward-Guard':  s = {'SG','SF'} | ({'PF'} if h >= 80 else set())
    elif p == 'Forward':        s = {'SF','PF'} if h <= 82 else {'PF'}
    elif p == 'Forward-Center': s = {'PF','C'}  | ({'SF'} if h <= 82 else set())
    elif p == 'Center-Forward': s = {'PF','C'}
    elif p == 'Center':         s = {'C'}
    else:                       s = set()
    if not s:      # unknown label -> fall back to height alone
        s = ({'PG','SG'} if h <= 75 else {'SG','SF'} if h <= 78
             else {'SF','PF'} if h <= 81 else {'PF','C'})
    return s

df_meta['AGE'] = ((DRAFT_DATE - pd.to_datetime(df_meta['BIRTH_DATE'])).dt.days / 365.25).round(1)
df_meta['POS_ELIGIBLE'] = [expand_position(p, h)
                           for p, h in zip(df_meta['POSITION'], df_meta['HEIGHT_IN'])]
df_meta['POS_STR'] = [''.join(sorted(s, key=ALL_POSITIONS.index)) for s in df_meta['POS_ELIGIBLE']]

print(f'\n✅ metadata for {len(df_meta)} players')
print(df_meta[['PLAYER_ID','POSITION','HEIGHT_IN','AGE','POS_STR']].head(8).to_string(index=False))

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# CELL 5 — CURRENT 2026-27 ROSTERS (disk-cached)
# ═══════════════════════════════════════════════════════════════════
# The game logs reflect each player's OLD team. The 2026 draft, trades and summer-2026 free
# agency reshuffled rosters, and usage is a function of who else is on the floor -- so Cell 6's
# teammate-strength adjustment needs to know where everyone plays NOW.

ROSTER_FILE = f'{CACHE_DIR}/team_rosters_2026-27.csv'
if os.path.exists(ROSTER_FILE):
    df_rosters = pd.read_csv(ROSTER_FILE)
    print(f'📦 rosters from cache: {len(df_rosters)} players, '
          f"{df_rosters['TEAM_ABBREVIATION'].nunique()} teams")
else:
    out = []
    for i, t in enumerate(static_teams.get_teams()):      # 30 static teams, no API call
        try:
            r = commonteamroster.CommonTeamRoster(
                    team_id=t['id'], season='2026-27').get_data_frames()[0]
            r['TEAM_ABBREVIATION'] = t['abbreviation']
            out.append(r[['PLAYER_ID','PLAYER','TEAM_ID','TEAM_ABBREVIATION']])
            time.sleep(0.8)
        except Exception as e:
            print(f"  warn {t['abbreviation']}: {e}")
        if (i+1) % 10 == 0 and out:
            pd.concat(out, ignore_index=True).to_csv(ROSTER_FILE, index=False)
            print(f'  checkpoint: {i+1}/30 teams'); time.sleep(3.0)
    df_rosters = pd.concat(out, ignore_index=True)
    df_rosters.to_csv(ROSTER_FILE, index=False)

print(f"✅ {len(df_rosters)} players on {df_rosters['TEAM_ABBREVIATION'].nunique()} rosters")

---
## Cell 5B · Real ESPN market data — auction values and true position eligibility

Two of the weakest assumptions in this notebook can be replaced with ESPN's own data.

**The league page you actually want is private.** `fantasy.espn.com/.../editdraftstrategy?leagueId=…`
requires your logged-in ESPN session, and the league API returns **401** without your
`espn_s2`/`SWID` cookies. What *is* public is ESPN's player universe for season 2027, and it
carries the two fields that matter:

| field | replaces | why it matters |
|---|---|---|
| `ownership.auctionValueAverage` | the VAR-derived `MARKET_$` | the **actual** price the market pays, rather than my model's guess at it |
| `eligibleSlots` | the height-based position heuristic | ESPN's **real** PG/SG/SF/PF/C eligibility |

### Why real positions matter more than they sound

The height heuristic in Cell 4 disagrees with ESPN for roughly **60%** of the pool, and not
harmlessly. ESPN lists Luka Dončić as **PG only** (the heuristic guessed SG/SF/PF), Wembanyama as
**C only**, and — the expensive one — **Giannis as PF/C**, meaning he counts against the
4-Center roster cap. Getting these wrong means building rosters that ESPN would reject and
mis-costing the Center constraint that this whole strategy runs into.

### Why market prices matter even more

`MARKET_$` is supposed to mean "what the room will pay," and the room does not use my model. Where
ESPN and my model disagree, **the disagreement is the edge**: my model prices Jamal Murray at $47
against ESPN's $24, and Giannis at $30 against ESPN's $63 (my convex durability penalty punishes
his 52-game projection; ESPN's does not). Buying against ESPN's number is what `SURPLUS` is
supposed to measure, and it only means something if the market side is real.

### One judgement call: rescaling

ESPN's values for the top 182 players sum to about **$1,931**, but this league puts **$2,800** on
the table (14 × $200). ESPN publishes observed averages from real drafts, which run short of a
full-spend allocation. Since every simulation here assumes teams spend their budgets, the values
are rescaled so the top `TOTAL_DRAFTED` sum to the league pot, preserving ESPN's *relative*
ordering — which is the part that carries the market information. Set
`ESPN_SCALE_TO_CAP = False` to use ESPN's raw numbers instead.

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# CELL 5B — ESPN MARKET DATA: auction values + true position eligibility
# ═══════════════════════════════════════════════════════════════════
# Public endpoint (no login). The private league endpoint
#   lm-api-reads.fantasy.espn.com/.../leagues/1015252784
# returns 401 without your espn_s2 / SWID cookies, so these are ESPN's
# league-agnostic values, rescaled below to this league's cap.
import json, urllib.request, unicodedata

ESPN_SEASON   = 2027
ESPN_CACHE    = f'{CACHE_DIR}/espn_market_{ESPN_SEASON}.json'
ESPN_URL      = (f'https://lm-api-reads.fantasy.espn.com/apis/v3/games/fba/'
                 f'seasons/{ESPN_SEASON}/players?view=kona_player_info')
ESPN_FILTER   = '{"players":{"limit":1500,"sortPercOwned":{"sortPriority":1,"sortAsc":false}}}'
USE_ESPN_MARKET    = True    # False -> fall back to this notebook's own VAR price model
USE_ESPN_POSITIONS = True    # False -> fall back to the Cell 4 height heuristic
# ESPN's raw values are calibrated for a smaller default league: for our pool the top 182 of them
# sum to about $1,832, while this league puts $2,800 on the table. Leaving them raw means only
# $1,832 of listed value is chasing $2,800 of budget, so the $200 cap stops binding -- the roster
# optimiser then buys a monstrous team for $200 and EVERY plan's win probability inflates
# (balanced measured 0.80 instead of ~0.50, which is nonsense). Prices in a market where money
# exceeds listed value clear ABOVE list, and this rescale is that inflation: it preserves ESPN's
# relative ordering, which is where the market information lives, and makes the budget bind.
#
# It does mean elite prices look steep against ESPN's UI (Jokic ~$134 rather than $85). That is
# the honest implication of a 14-team full-spend auction, not a bug -- deeper leagues lower
# replacement level and bid stars up. The raw ESPN number is kept alongside as ESPN_$_LIST so you
# can see both on draft night. Set this False to bid strictly off ESPN's published numbers.
ESPN_SCALE_TO_CAP  = True

if os.path.exists(ESPN_CACHE):
    espn_raw = json.load(open(ESPN_CACHE))
    print(f'📦 ESPN market data from cache: {len(espn_raw)} players')
else:
    print('🌐 fetching ESPN player universe ...', end='', flush=True)
    req = urllib.request.Request(ESPN_URL, headers={
        'X-Fantasy-Filter': ESPN_FILTER,
        'Accept': 'application/json',
        'User-Agent': 'Mozilla/5.0',
    })
    with urllib.request.urlopen(req, timeout=60) as r:
        espn_raw = json.loads(r.read().decode())
    with open(ESPN_CACHE, 'w') as f:
        json.dump(espn_raw, f)
    print(f' {len(espn_raw)} players → cached')

# ESPN lineup-slot ids -> the five atomic positions this notebook reasons about.
# (5=G, 6=F, 7=SG/SF, 8=G/F, 9=PF/C, 10=F/C, 11=UTIL, 12=BENCH, 13=IR are all derived
#  from these five, so only 0-4 are needed.)
ESPN_SLOT = {0: 'PG', 1: 'SG', 2: 'SF', 3: 'PF', 4: 'C'}
ESPN_DEFAULT_POS = {1: 'PG', 2: 'SG', 3: 'SF', 4: 'PF', 5: 'C'}

SUFFIXES = (' jr', ' sr', ' ii', ' iii', ' iv', ' v')
def name_key(s):
    """'Nikola Jokić' -> 'nikolajokic'. ESPN strips accents and sometimes suffixes, so match on
    an accent-free, punctuation-free, suffix-free key."""
    s = ''.join(c for c in unicodedata.normalize('NFKD', str(s)) if not unicodedata.combining(c))
    s = s.lower().replace('.', ' ').replace('-', ' ').replace("'", ' ')
    s = ' '.join(s.split())
    for suf in SUFFIXES:
        if s.endswith(suf):
            s = s[:-len(suf)]
            break
    return ''.join(ch for ch in s if ch.isalnum())

rows = []
for p in espn_raw:
    own = p.get('ownership') or {}
    slots = set(p.get('eligibleSlots') or ())
    pos = {ESPN_SLOT[i] for i in ESPN_SLOT if i in slots}
    rows.append({
        'KEY': name_key(p['fullName']),
        'ESPN_NAME': p['fullName'],
        'ESPN_$_RAW': own.get('auctionValueAverage'),
        'ESPN_PCT_OWNED': own.get('percentOwned'),
        'ESPN_POS_SET': pos if pos else None,
        'ESPN_PRIMARY': ESPN_DEFAULT_POS.get(p.get('defaultPositionId')),
        'ESPN_INJURY': p.get('injuryStatus'),
    })
df_espn = (pd.DataFrame(rows)
             .sort_values('ESPN_$_RAW', ascending=False, na_position='last')
             .drop_duplicates('KEY'))

n_val = df_espn['ESPN_$_RAW'].notna().sum()
top_sum = df_espn['ESPN_$_RAW'].nlargest(TOTAL_DRAFTED).sum()
print(f'   {n_val} players carry an ESPN auction value; '
      f'{df_espn["ESPN_POS_SET"].notna().sum()} carry eligible slots')
print(f'   ESPN top-{TOTAL_DRAFTED} values sum to ${top_sum:,.0f}; this league puts '
      f'${TOTAL_BUDGET:,} on the table')
print(f'   scale to cap: {ESPN_SCALE_TO_CAP}  ·  use ESPN market: {USE_ESPN_MARKET}'
      f'  ·  use ESPN positions: {USE_ESPN_POSITIONS}')
print('\n   ESPN market top 10:')
print(df_espn.head(10)[['ESPN_NAME','ESPN_$_RAW','ESPN_PRIMARY','ESPN_PCT_OWNED']]
        .to_string(index=False))

---
## Cell 6 · Projections — and why the **standard deviation** is the important output

Standard projection mechanics: blend the two seasons 70/30, apply an age curve, adjust for
teammate strength on the new 2026-27 roster, and derive expected availability from games
played.

The part that is *not* standard, and that the whole strategy rests on: this cell keeps each
player's **per-game standard deviation** alongside the mean.

A mean alone cannot answer the only question that matters here. Two players averaging 1.4
steals are completely different assets if one posts 1–2 every night and the other alternates
0 and 4. Over a 3–4 game week those produce very different *distributions* of team totals,
and a category is won or lost on the distribution, not the average. Because
$\sigma_{\text{week}} = \sqrt{g}\,\sigma_{\text{game}}$, a weekly total is far less volatile
than a single game — which is exactly why **a category can be locked at all**, and why the
lock is stronger for players who play more games.

Three adjustments, each deliberately bounded:

- **Age curve** — peak 23–28, growth below, decline above. Applied to production means, not to variance.
- **Teammate context** (±15% cap) — the same player produces less surrounded by other stars. Applied only to *usage-driven counting stats*, never to FG%/FT%, which are shooting skill rather than role. The cap is there because these roster combinations have never played a game together; anything less bounded would be false precision.
- **Availability** — mean GP across the two seasons, which feeds the convex durability weight from Cell 2.

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# CELL 6 — PROJECTIONS WITH PER-GAME VARIANCE
# ═══════════════════════════════════════════════════════════════════
# Raw per-game quantities we track. MIN/FGM/FGA/FTM/FTA are not categories themselves, but
# the weekly ratio categories are built from their team totals, so they must be simulated.
RAW = ['MIN','PTS','REB','AST','STL','BLK','FG3M','FGM','FGA','FTM','FTA','DD']

GP_MIN, MPG_MIN = 15, 12        # rosterable-player filter

def season_agg(df):
    # Per-game MEAN and STD for every raw quantity -> the variance model's input.
    gb = df.groupby(['PLAYER_ID','PLAYER_NAME'])
    g = pd.concat([gb['GAME_ID'].count().rename('GP'),
                   gb[RAW].mean().add_suffix('_M'),
                   gb[RAW].std().add_suffix('_S')], axis=1).reset_index()
    g['MPG'] = g['MIN_M']
    g = g[(g.GP >= GP_MIN) & (g.MPG >= MPG_MIN)].copy()
    for c in RAW:
        g[c+'_S'] = g[c+'_S'].fillna(0.0)
    return g

def age_factor(a):
    if pd.isna(a): return 1.0
    return (1.14 if a < 21 else 1.09 if a < 23 else 1.04 if a < 25 else 1.00 if a <= 28
            else 0.97 if a <= 30 else 0.92 if a <= 32 else 0.86 if a <= 34 else 0.80)

a_cur  = season_agg(df_logs[df_logs.SEASON == CURRENT_SEASON])
a_prev = season_agg(df_logs[df_logs.SEASON == PREV_SEASON])
P = a_cur.merge(a_prev, on=['PLAYER_ID','PLAYER_NAME'], how='left', suffixes=('','_P'))

# ── blended mean; std from the current season (recency = best variance estimate) ──
for c in RAW:
    cu, pv = P[c+'_M'], P[c+'_M_P']
    P['P_'+c+'_M'] = np.where(pv.notna(), W_CURRENT*cu + W_PREV*pv, cu)
    P['P_'+c+'_S'] = P[c+'_S']

# ── age / height / position ──
P = P.merge(df_meta[['PLAYER_ID','AGE','HEIGHT_IN','POSITION','POS_ELIGIBLE','POS_STR']],
            on='PLAYER_ID', how='left')
P['AGE_FACTOR'] = P['AGE'].apply(age_factor).fillna(1.0)
P['HEIGHT_IN']  = P['HEIGHT_IN'].fillna(78)
P['POS_ELIGIBLE'] = [p if isinstance(p, set) else {'SG','SF'} for p in P['POS_ELIGIBLE']]

# ── ESPN's real position eligibility replaces the height heuristic (Cell 5B) ──
# The heuristic disagrees with ESPN for ~60% of the pool, and the misses are costly: ESPN has
# Doncic as PG-only, Wembanyama as C-only, and Giannis as PF/C -- so Giannis counts against the
# 4-Center roster cap, which the heuristic never knew. Roster legality and the Center constraint
# both depend on getting this right, so ESPN wins wherever it has an answer.
P['KEY'] = P['PLAYER_NAME'].map(name_key)
P = P.merge(df_espn[['KEY','ESPN_NAME','ESPN_$_RAW','ESPN_POS_SET','ESPN_PRIMARY',
                     'ESPN_INJURY','ESPN_PCT_OWNED']], on='KEY', how='left')
P['POS_SOURCE'] = 'height heuristic'
if USE_ESPN_POSITIONS:
    use_espn = P['ESPN_POS_SET'].notna()
    n_changed = sum(1 for ok, a, b in zip(use_espn, P['POS_ELIGIBLE'], P['ESPN_POS_SET'])
                    if ok and set(a) != set(b))
    P.loc[use_espn, 'POS_SOURCE'] = 'ESPN'
    P['POS_ELIGIBLE'] = [set(b) if ok else a
                         for ok, a, b in zip(use_espn, P['POS_ELIGIBLE'], P['ESPN_POS_SET'])]
    print(f'📍 positions: {int(use_espn.sum())}/{len(P)} from ESPN '
          f'({n_changed} differed from the height heuristic), '
          f'{int((~use_espn).sum())} left on the heuristic')
P['POS_STR'] = [''.join(sorted(s, key=ALL_POSITIONS.index)) for s in P['POS_ELIGIBLE']]
n_c = sum(1 for s in P['POS_ELIGIBLE'] if 'C' in s)
print(f'   Center-eligible players in the pool: {n_c} '
      f'(the {MAX_CENTERS}-C roster cap is priced against this)')

# ── availability -> convex durability weight (Cell 2 derived the exponent) ──
gp_hist = (df_logs.groupby(['PLAYER_ID','SEASON'])['GAME_ID'].count()
                  .groupby('PLAYER_ID').mean().rename('AVG_GP').reset_index())
P = P.merge(gp_hist, on='PLAYER_ID', how='left')
P['AVAIL']  = (P['AVG_GP'] / MAX_GAMES).clip(0.20, 1.0)
P['EXP_GP'] = (P['AVAIL'] * MAX_GAMES).round(0)
P['DUR_W']  = P['AVAIL'] ** DURABILITY_EXPONENT
P['RISK'] = np.select([P.AVAIL >= 0.90, P.AVAIL >= 0.78, P.AVAIL >= 0.65],
                      ['IRON MAN', 'STABLE', 'SOME RISK'], default='FRAGILE')

# ── current 2026-27 team (fall back to last 2025-26 team) ──
last_team = (df_logs[df_logs.SEASON == CURRENT_SEASON].sort_values('GAME_DATE')
             .groupby('PLAYER_ID')['TEAM_ABBREVIATION'].last().rename('TEAM_OLD').reset_index())
P = (P.merge(df_rosters[['PLAYER_ID','TEAM_ABBREVIATION']].rename(
                 columns={'TEAM_ABBREVIATION':'TEAM'}), on='PLAYER_ID', how='left')
       .merge(last_team, on='PLAYER_ID', how='left'))
P['ON_ROSTER_2627'] = P['TEAM'].notna()
P['TEAM_SOURCE'] = np.where(P['ON_ROSTER_2627'], '2026-27 roster', 'not rostered')
P['TEAM'] = P['TEAM'].fillna(P['TEAM_OLD']).fillna('FA')

# ── DRAFTABILITY: a player who is not on an NBA roster cannot be drafted ─────
# The 2025-26 game logs contain players who have since retired or gone unsigned -- Russell
# Westbrook, Kevin Love, Jonas Valanciunas, Lonzo Ball and others all posted real 2025-26
# numbers and would otherwise be priced, recommended, and "drafted" by the simulations.
# Falling back to a player's last 2025-26 team (which the line above does) is right for the
# teammate-context calculation but must NOT make him draftable.
#
# Set REQUIRE_ON_ROSTER = False if the Cell 5 roster fetch looks incomplete (e.g. it errored
# for some teams) and you would rather keep everyone than lose real players.
REQUIRE_ON_ROSTER = True
if REQUIRE_ON_ROSTER:
    dropped = P.loc[~P['ON_ROSTER_2627'], ['PLAYER_NAME', 'TEAM_OLD', 'P_PTS_M']].copy()
    P = P[P['ON_ROSTER_2627']].copy()
    print(f'🚫 removed {len(dropped)} players not on any 2026-27 roster '
          f'(retired / unsigned) — they are not draftable:')
    for _, r in dropped.sort_values('P_PTS_M', ascending=False).head(12).iterrows():
        print(f"     {r['PLAYER_NAME']:<24} (last: {r['TEAM_OLD']}, {r['P_PTS_M']:.1f} pts/g)")
    if len(dropped) > 12:
        print(f"     ... and {len(dropped)-12} more")

# age applies to production, not to variance
for c in ['PTS','REB','AST','STL','BLK','FG3M','FGM','FGA','FTM','FTA','DD','MIN']:
    P['P_'+c+'_M'] *= P['AGE_FACTOR']

# ── teammate strength -> bounded usage adjustment on counting stats only ──
star = np.zeros(len(P))
for c in ['PTS','REB','AST','STL','BLK','FG3M']:
    v = P['P_'+c+'_M']; star += ((v - v.mean()) / v.std()).to_numpy()
P['STAR_SCORE'] = star
P['MATES'] = P.groupby('TEAM')['STAR_SCORE'].transform('sum') - P['STAR_SCORE']
mz = (P['MATES'] - P['MATES'].mean()) / P['MATES'].std()
P['CTX'] = (1 - 0.06*mz).clip(0.85, 1.15)      # crowded roster -> less usage available
for c in ['PTS','REB','AST','STL','BLK','FG3M','FGM','FGA','FTM','FTA','DD']:
    P['P_'+c+'_M'] *= P['CTX']

P = P.reset_index(drop=True)
print(f'✅ {len(P)} rosterable players projected (GP>={GP_MIN}, MPG>={MPG_MIN})')
print(f'   availability: mean {P.AVAIL.mean():.2f}  ·  context factor range '
      f'{P.CTX.min():.2f}-{P.CTX.max():.2f}')
print('\nTop 10 projected scorers — note MEAN vs per-game SD (the SD is what decides locks):')
show = P.nlargest(10, 'P_PTS_M')[['PLAYER_NAME','TEAM','POS_STR','AGE','EXP_GP','RISK',
                                  'P_PTS_M','P_PTS_S','P_REB_M','P_AST_M','P_STL_M','P_DD_M']]
print(show.round(2).to_string(index=False))

---
## Cell 7 · Category structure — *why* some categories are cheap to dominate

This is the analytical foundation. Before choosing categories, measure the two properties
that decide how expensive a category is to win.

**Correlation.** If two categories are driven by the same kind of player, one purchase pays
into both. Targeting two highly-correlated categories costs barely more than targeting one;
targeting two *anti*-correlated categories means every dollar fights itself.

**Concentration.** If the league's supply of a category is spread evenly across 200 players,
you must beat the field everywhere to win it. If 20 players own most of the supply, **two or
three buys win the category outright** — and everyone else's spending on it is wasted. This
is the property that actually decides where the bargains are, and it is the one most draft
strategies never look at.

The metrics below:

- **TOP20_SHARE** — share of all rosterable production held by the top 20 players. Higher = more concentrated = cheaper to dominate.
- **GINI** — inequality of supply across the pool, 0 = perfectly even, 1 = one player owns everything.
- **CV** — coefficient of variation across the pool; how far the elite sit above the median.

Read the correlation matrix for *clusters*, and the concentration table for *leverage*.
The two together determine the shortlist that Cell 11 searches over.

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# CELL 7 — CATEGORY STRUCTURE: correlation + concentration
# ═══════════════════════════════════════════════════════════════════
# Per-player category rates, as a manager would read them off a projection page.
def player_cat_rates(P):
    d = pd.DataFrame({'PLAYER_ID': P['PLAYER_ID'], 'PLAYER_NAME': P['PLAYER_NAME']})
    for c in COUNTING_CATS:
        d[c] = P['P_'+c+'_M']
    d['FG_PCT']  = np.where(P['P_FGA_M'] > 0, P['P_FGM_M'] / P['P_FGA_M'], np.nan)
    d['FT_PCT']  = np.where(P['P_FTA_M'] > 0, P['P_FTM_M'] / P['P_FTA_M'], np.nan)
    d['PTS_MIN'] = np.where(P['P_MIN_M'] > 0, P['P_PTS_M'] / P['P_MIN_M'], np.nan)
    # volumes: needed because a ratio only matters in proportion to attempts
    d['FGA'] = P['P_FGA_M']; d['FTA'] = P['P_FTA_M']; d['MIN'] = P['P_MIN_M']
    return d

rates = player_cat_rates(P)

# ── 1. correlation between categories, across the rosterable pool ──
Zraw = rates[CATS].apply(lambda s: (s - s.mean()) / s.std())
corr = Zraw.corr()
corr.index = [LBL[c] for c in corr.index]; corr.columns = [LBL[c] for c in corr.columns]
print('CATEGORY CORRELATION  (x100, across the rosterable pool)')
print('   read for CLUSTERS: categories that move together are cheap to buy together')
print((corr*100).round(0).astype(int).to_string())

# ── 2. concentration: how top-heavy is each category's supply? ──
# Measured in the units that actually move a TEAM total (Cell 8's "impact"), so counting and
# ratio categories are directly comparable: EDGE = how many standard deviations of category
# advantage the elite players hand you. High EDGE_TOP3 means a couple of buys win the
# category outright -- which is exactly what makes a category cheap to dominate.
def gini(x):
    x = np.sort(np.clip(np.asarray(x, float), 0, None))
    n, s = len(x), x.sum()
    return float(((2*np.arange(1, n+1) - n - 1) * x).sum() / (n*s)) if s > 0 else 0.0

def impact_units(rates):
    out = {}
    for c in COUNTING_CATS:
        out[c] = rates[c].to_numpy(float)
    for c, vol in [('FG_PCT','FGA'), ('FT_PCT','FTA'), ('PTS_MIN','MIN')]:
        lg = (rates[vol] * rates[c]).sum() / rates[vol].sum()
        out[c] = ((rates[c] - lg) * rates[vol]).to_numpy(float)
    return out

_imp = impact_units(rates)
conc = []
for c in CATS:
    v = _imp[c]; v = v[~np.isnan(v)]
    z = (v - v.mean()) / v.std()
    top = np.sort(z)[::-1]
    conc.append({'CAT': LBL[c],
                 'EDGE_TOP3':  top[:3].mean(),
                 'EDGE_TOP20': top[:20].mean(),
                 'GINI': gini(v)})
df_conc = pd.DataFrame(conc).sort_values('EDGE_TOP3', ascending=False)

print('\n\nCATEGORY CONCENTRATION  —  how much edge the elite hand you, in pool sigmas')
print('   EDGE_TOP3/20 = mean z-score of the best 3 / best 20 assets in that category')
print('   higher = fewer buys needed to own the category = CHEAP to dominate')
print(df_conc.round(2).to_string(index=False))
print(f"\n   Cheapest to dominate : {' > '.join(df_conc.head(3).CAT)}"
      f"  <- a few buys can win these outright")
print(f"   Most expensive       : {' > '.join(df_conc.tail(3).CAT)}"
      f"  <- edge is spread thin, you must outspend the field")

---
## Cell 8 · Valuation — market price vs *your* price

Two different numbers, and conflating them is the most expensive mistake available in an
auction.

**Market price** is what the room will pay. The room prices players on balanced 10-category
value, so market price is computed from the full-category z-score sum, converted to dollars
by **value above replacement**: replacement level is the 182nd-best player (14 teams × 13
spots), every player's surplus over that is summed, and the $2,800 of league money above the
$1 minimums is distributed in proportion. That is the standard model and it is the right one
here, because it describes *the room*, not you.

**Your price** is the same computation restricted to your target categories. A player who is
elite in five of your seven targets and useless in the three you punt is worth far more to you
than to the room.

**`SURPLUS = your price − market price` is the entire edge.** It is the list of players you
can buy below what they are worth *to you*, and it is why a punt build can beat a balanced
build that spent the same $200.

### The volume-weighting fix on ratio categories

For counting categories a z-score on per-game production is correct. For **FG%, FT% and PPM**
it is not, and this is worth being precise about, because the naive version actively
misleads:

A team's FT% is $\Sigma FTM / \Sigma FTA$. A player changes it by
$(\text{his rate} - \text{league rate}) \times \text{his attempts}$. So a 92% shooter taking
1 attempt a game moves your team FT% by almost nothing, while an 80% shooter taking 8
attempts moves it a great deal. **Z-scoring the raw percentage ranks the first player as
elite and the second as mediocre — exactly backwards.** This cell z-scores the
volume-weighted impact instead, which materially re-orders the FG%/FT%/PPM boards and changes
who is worth buying.

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# CELL 8 — VALUATION: volume-weighted z-scores, market price, your price
# ═══════════════════════════════════════════════════════════════════
def category_impact(rates):
    # Per-category contribution in the units that actually move a TEAM total.
    # Counting cats: per-game production. Ratio cats: (rate - league rate) * volume.
    imp = pd.DataFrame(index=rates.index)
    for c in COUNTING_CATS:
        imp[c] = rates[c]
    lg = {}
    for c, vol in [('FG_PCT','FGA'), ('FT_PCT','FTA'), ('PTS_MIN','MIN')]:
        lg[c] = (rates[vol] * rates[c]).sum() / rates[vol].sum()
        imp[c] = (rates[c] - lg[c]) * rates[vol]
    return imp, lg

impact, LEAGUE_RATE = category_impact(rates)
Z = pd.DataFrame({'Z_'+c: (impact[c] - impact[c].mean()) / impact[c].std() for c in CATS})
P = pd.concat([P, Z], axis=1)

def strategy_value(Z, targets, dur_w):
    # Value under a punt plan: only the target categories count, scaled by the convex
    # durability weight -- an absent player posts zeros in a locked weekly lineup.
    return sum(Z['Z_'+c] for c in targets) * dur_w

def auction_prices(value, n_drafted=TOTAL_DRAFTED, budget=TOTAL_BUDGET, min_bid=MIN_BID):
    # Value above replacement -> dollars. Replacement = the n_drafted-th best player.
    v = np.asarray(value, float)
    order = np.argsort(-v)
    repl = v[order[min(n_drafted, len(v)) - 1]]
    var = np.clip(v - repl, 0, None)
    pot = budget - n_drafted*min_bid
    tot = var[order[:n_drafted]].sum()
    if tot <= 0:
        return np.full(len(v), min_bid, int)
    drafted = np.zeros(len(v), bool); drafted[order[:n_drafted]] = True
    return np.where(drafted, np.round((var/tot)*pot + min_bid), 0).astype(int)

# ── MARKET price: what the room will actually pay ──
# First choice is ESPN's own auction values (Cell 5B) -- the market's real prices, not a model's
# guess at them. The VAR model is still computed as MODEL_$ so the two can be compared: where
# they disagree is where the bargains are, and that comparison is printed below.
P['VAL_MARKET'] = strategy_value(Z, CATS, P['DUR_W'])
P['MODEL_$']    = auction_prices(P['VAL_MARKET'])

if USE_ESPN_MARKET and P['ESPN_$_RAW'].notna().any():
    espn = P['ESPN_$_RAW'].astype(float)
    if ESPN_SCALE_TO_CAP:
        # Preserve ESPN's relative ordering, but make the top TOTAL_DRAFTED sum to the league
        # pot: ESPN publishes observed averages that fall short of a full-spend auction, and
        # every simulation here assumes teams spend their budgets.
        top = espn.nlargest(TOTAL_DRAFTED)
        excess_espn = max(top.sum() - len(top)*MIN_BID, 1.0)
        excess_ours = TOTAL_BUDGET - TOTAL_DRAFTED*MIN_BID
        scale = excess_ours / excess_espn
        scaled = MIN_BID + (espn - MIN_BID).clip(lower=0) * scale
    else:
        scale, scaled = 1.0, espn
    # players ESPN does not price (deep bench, two-ways) fall back to the model
    P['ESPN_$_LIST'] = P['ESPN_$_RAW'].round()      # what ESPN's own UI shows you
    P['MARKET_$'] = np.where(scaled.notna(), scaled.round(), P['MODEL_$']).astype(int)
    P['MARKET_$'] = np.maximum(P['MARKET_$'], 0)
    P['MARKET_SOURCE'] = np.where(scaled.notna(), 'ESPN', 'model')
    n_espn = int((P['MARKET_SOURCE'] == 'ESPN').sum())
    print(f'💰 MARKET_$ = ESPN auction values'
          + (f' × {scale:.2f}' if ESPN_SCALE_TO_CAP else ' (raw — already per $200 budget)')
          + f': {n_espn} players, {int((P["MARKET_SOURCE"]=="model").sum())} on the fallback model')
else:
    P['MARKET_$'] = P['MODEL_$']
    P['MARKET_SOURCE'] = 'model'
    print('💰 MARKET_$ = this notebook\'s VAR model (ESPN market data unavailable/disabled)')

_top = P['MARKET_$'].nlargest(TOTAL_DRAFTED).sum()
print(f"✅ market board priced. Top {TOTAL_DRAFTED} (the players who get drafted) = ${_top:,}"
      f" against a ${TOTAL_BUDGET:,} league pot.")
print(f"   Running ${TOTAL_BUDGET - _top:,} short of the cap is expected and is not an error:"
      f" real auctions\n   end with unspent budgets and $1 endgame buys. The Cell 15 auction"
      f" bids that slack back in.")
print(f"   league rates: FG% {LEAGUE_RATE['FG_PCT']:.3f} · FT% {LEAGUE_RATE['FT_PCT']:.3f} "
      f"· PPM {LEAGUE_RATE['PTS_MIN']:.3f}")
print('\nTOP 15 — MARKET BOARD (what the room will pay)')
print(P.nlargest(15,'MARKET_$')[['PLAYER_NAME','TEAM','POS_STR','AGE','EXP_GP','RISK',
                                 'MARKET_$','P_PTS_M','P_REB_M','P_AST_M']]
       .round(2).to_string(index=False))

# ── where ESPN and the model disagree: the market's soft spots ──
if (P['MARKET_SOURCE'] == 'ESPN').any():
    cmpm = P[P['MARKET_SOURCE'] == 'ESPN'].copy()
    cmpm['GAP'] = cmpm['MARKET_$'] - cmpm['MODEL_$']
    CG = ['PLAYER_NAME','TEAM','POS_STR','AGE','EXP_GP','RISK','MARKET_$','MODEL_$','GAP']
    print('\n\nESPN MARKET vs THIS MODEL — the market pays UP for these (fade them):')
    print(cmpm.nlargest(10, 'GAP')[CG].to_string(index=False))
    print('\n   ...and the market DISCOUNTS these relative to the model (look here first):')
    print(cmpm.nsmallest(10, 'GAP')[CG].to_string(index=False))
    print('\n   Two causes, and the first one is the whole opportunity:')
    print('   1. CATEGORY MISMATCH. ESPN prices players for its default 9-cat scoring, which has'
          '\n      no DD and no PPM. This league scores both. So rebounding centres -- who pile up'
          '\n      REB, DD and FG% together -- are priced as if double-doubles were worth nothing'
          '\n      to anyone. That is a structural discount the room inherits from ESPN, and it is'
          '\n      exactly where a REB/DD plan shops.')
    print(f'   2. DURABILITY. The model applies a convex games-played penalty (exponent '
          f'{DURABILITY_EXPONENT});\n      ESPN does not, so fragile stars look dear to the model'
          ' and cheap to the room.')

# Demonstration of the volume-weighting fix on FT%
naive = (rates['FT_PCT'] - rates['FT_PCT'].mean()) / rates['FT_PCT'].std()
cmp = pd.DataFrame({'PLAYER': rates['PLAYER_NAME'], 'FT%': rates['FT_PCT'],
                    'FTA/g': rates['FTA'], 'NAIVE_Z': naive, 'VOLUME_WEIGHTED_Z': Z['Z_FT_PCT']})
print('\n\nWHY VOLUME-WEIGHTING MATTERS — top 8 by the NAIVE (raw-percentage) FT% z-score:')
print(cmp.nlargest(8,'NAIVE_Z').round(2).to_string(index=False))
print('\n   ...and the top 8 by volume-weighted FT% impact (who actually moves your team FT%):')
print(cmp.nlargest(8,'VOLUME_WEIGHTED_Z').round(2).to_string(index=False))

---
## Cell 9 · The weekly scoring engine — ESPN's actual rules

This cell is the heart of the notebook, and it is where most draft models quietly go wrong.

A weekly matchup in this league is settled like this:

- Only the **10 starters** score. Bench players contribute nothing.
- **Counting categories** are *summed over every game each starter plays that week*. A starter who plays 4 games contributes a third more than one who plays 3.
- **Ratio categories** are **team aggregates**: FG% is ΣFGM/ΣFGA over the whole lineup, PPM is ΣPTS/ΣMIN. They are *not* the average of ten individual percentages.

Three consequences follow, and each changes valuations:

**Games played is a category multiplier.** Every counting category scales with games, so
availability is not a tie-breaker — it is leverage on your entire target set at once. This is
the same reason Cell 2's durability exponent is convex.

**Ratio categories punish dead weight asymmetrically.** One high-minute, low-efficiency
starter drags PPM and FG% down for the whole team, no matter how good the other nine are.
That is a strong argument for *not* rostering minutes-heavy non-scorers if PPM is a target —
and an equally strong argument for ignoring efficiency entirely once you punt it.

**A weekly total is much steadier than a single game.** Since
$\sigma_{\text{week}} = \sqrt{g}\,\sigma_{\text{game}}$, a 3–4 game week smooths out
individual-night noise. This is *why a category can be locked at all* — and it is why the
margin needed for a lock is smaller than intuition suggests.

The simulator draws each starter's games from the week's team schedule filtered by his
availability, then draws his per-game production from his own mean and standard deviation.
`cat_win_probs` returns P(win) per category; `matchup_win_prob` returns P(win ≥ 6 of 10) —
the only number that actually matters. Exact ties are counted as half a win, because this
league has **no category tie-breakers**.

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# CELL 9 — WEEKLY MATCHUP SIMULATOR (real ESPN scoring)
# ═══════════════════════════════════════════════════════════════════
def sim_weeks(roster, n_weeks, rng):
    """Simulate n_weeks of ESPN H2H-each-category weekly totals for one lineup.

    Counting cats = summed over games played by the 10 STARTERS.
    Ratio cats    = team aggregates (SUM FGM / SUM FGA, SUM PTS / SUM MIN),
                    never a mean of per-player ratios.
    Returns {category: array of length n_weeks}.
    """
    st = roster[roster.ROLE == 'STARTER'] if 'ROLE' in roster.columns else roster
    k = len(st)
    avail = st['AVAIL'].to_numpy()[None, :]
    # team games in a 7-day matchup, then how many the player is actually available for
    team_games = rng.binomial(4, GAMES_PER_WEEK/4.0, size=(n_weeks, k))
    g = rng.binomial(team_games, avail)

    tot = {}
    for c in RAW:
        mu = st['P_'+c+'_M'].to_numpy()[None, :]
        sd = st['P_'+c+'_S'].to_numpy()[None, :]
        # sum of g i.i.d. per-game draws: mean g*mu, sd sqrt(g)*sd
        tot[c] = np.clip(g*mu + np.sqrt(g)*sd*rng.standard_normal((n_weeks, k)), 0, None).sum(axis=1)

    out = {c: tot[c] for c in COUNTING_CATS}
    z = np.zeros(n_weeks)
    out['FG_PCT']  = np.divide(tot['FGM'], tot['FGA'], out=z.copy(), where=tot['FGA'] > 0)
    out['FT_PCT']  = np.divide(tot['FTM'], tot['FTA'], out=z.copy(), where=tot['FTA'] > 0)
    out['PTS_MIN'] = np.divide(tot['PTS'], tot['MIN'], out=z.copy(), where=tot['MIN'] > 0)
    return out

def cat_win_probs(mine, theirs):
    # P(win each category) in a given week. No tie-breakers in this league -> tie = half.
    return {c: float((mine[c] > theirs[c]).mean() + 0.5*(mine[c] == theirs[c]).mean())
            for c in CATS}

def matchup_win_prob(mine, theirs):
    # THE objective: P(win at least 6 of 10 categories).
    w = np.zeros(len(mine['PTS']))
    for c in CATS:
        w += (mine[c] > theirs[c]).astype(float) + 0.5*(mine[c] == theirs[c])
    return float((w > CATS_TO_WIN - 1).mean() + 0.5*(w == CATS_TO_WIN - 1).mean())

def cats_won_dist(mine, theirs):
    # Distribution of how many categories you win -- shows the shape of the win, not just P.
    w = np.zeros(len(mine['PTS']))
    for c in CATS:
        w += (mine[c] > theirs[c]).astype(float) + 0.5*(mine[c] == theirs[c])
    return w

print('✅ weekly simulator ready')
print('   counting cats summed over games · ratio cats aggregated over the lineup')
print(f'   a matchup is won by taking {CATS_TO_WIN} of {len(CATS)} categories; ties count half')

---
## Cell 10 · Roster optimiser

Given a set of target categories and the market prices, build the best legal 13-man roster for
$200. The constraints are all real ESPN rules:

1. Exactly **13** players, total spend **≤ $200** (every player costs at least $1).
2. The 10 typed starter slots must all be fillable: **PG · SG · SF · PF · C · G · F · UTIL ×3**.
3. At most **4 Center-eligible** players on the roster, starters and bench combined.

Two design choices worth explaining:

**It maximises the value of the 10 who START, not of all 13.** Bench players never score in
this format. Their only jobs are to satisfy the roster and be streaming fodder under the
7-acquisitions-per-matchup rule, so the optimiser spends as close to nothing on them as the
rules allow. This is why the recommended rosters end with $0–1 bench players, and that is
correct rather than lazy.

**Construction is slot-driven, then improved by local search.** Walking the slots from
tightest (PG/SG/SF/PF/C) to loosest (UTIL) and reserving $1 for every unfilled spot
guarantees a legal roster by construction. A naive "take the best player available while the
roster stays legal" loop deadlocks: after nine guards, *no* remaining player can complete a
legal lineup, because filling the C slot still leaves SF/PF/F unfillable — so it rejects
everyone and returns a short roster. After construction, a swap pass trades any rostered
player for an unrostered one that raises started-lineup value while keeping the roster legal.
Both a stars-and-scrubs greedy and a value-per-dollar greedy are run, and the better result is
kept.

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# CELL 10 — ROSTER OPTIMISER (budget · ESPN slots · max 4 C)
# ═══════════════════════════════════════════════════════════════════
def match_starters(idx, val, eligs):
    """Assign the 10 typed ESPN starter slots to players from roster `idx`.

    Uses augmenting-path (Kuhn) bipartite matching with players tried in descending value
    order. Matching is required rather than greedy: filling each slot with its best eligible
    player strands slots. Example -- give slot PG the roster's best PG/SG-eligible player and
    slot G may be left with nobody, even though swapping the two would have filled both.
    Augmenting paths re-shuffle earlier assignments instead of giving up, so all 10 slots get
    filled whenever any legal assignment exists.

    Returns {slot_index: player_index} (possibly partial if the roster is genuinely illegal).
    """
    order = sorted(idx, key=lambda i: -val[i])
    owner = {}                                       # player -> slot
    filled = {}                                      # slot -> player
    def aug(si, seen):
        for pi in order:
            if pi in seen or not (eligs[pi] & SLOT_MASKS[si]):
                continue
            seen.add(pi)
            if pi not in owner or aug(owner[pi], seen):
                owner[pi] = si; filled[si] = pi
                return True
        return False
    # tightest slots first (PG/SG/SF/PF/C), then G/F, then UTIL
    for si in sorted(range(TOTAL_STARTERS), key=lambda s: len(SLOT_MASKS[s])):
        aug(si, set())
    return filled

def slots_fillable(eligs):
    """Can these eligibility sets fill all 10 typed starter slots? Only meaningful on a
    COMPLETE roster -- a 9-man roster cannot fill 10 slots, which is why roster construction
    below is slot-driven rather than testing partial rosters."""
    if len(eligs) < TOTAL_STARTERS:
        return True
    val = np.zeros(len(eligs))
    return len(match_starters(range(len(eligs)), val, list(eligs))) == TOTAL_STARTERS

def best_started_value(idx, val, eligs):
    """Value of the best legal 10-man starting lineup out of a 13-man roster."""
    filled = match_starters(idx, val, eligs)
    return float(sum(val[pi] for pi in filled.values()))

class Board:
    """Array-backed view of the player pool: fast enough to optimise 600+ times."""
    def __init__(self, df, value_col, price_col):
        self.df = df.reset_index(drop=True)
        self.val = self.df[value_col].fillna(-1e9).to_numpy(float)
        self.price = self.df[price_col].clip(lower=MIN_BID).to_numpy(int)
        self.eligs = list(self.df['POS_ELIGIBLE'])
        self.pid = self.df['PLAYER_ID'].to_numpy()
        self.isC = np.array(['C' in e for e in self.eligs])
        self.value_col, self.price_col = value_col, price_col

def build_roster(pool, value_col, price_col, budget=BUDGET_PER_TEAM, exclude=None,
                 board=None, n_cand=260, passes=10):
    # n_cand / passes trade optimisation quality for speed. The 628-bundle search in Cell 11
    # uses smaller values (it only needs a fair RANKING); the final roster uses the defaults.
    b = board if board is not None else Board(pool, value_col, price_col)
    keep = np.ones(len(b.val), bool)
    if exclude is not None and len(exclude):
        keep = ~np.isin(b.pid, list(exclude))
    live = np.flatnonzero(keep)
    if len(live) < ROSTER_SIZE:
        raise RuntimeError('player pool exhausted')

    def greedy(key):
        # Walk slots tightest-first, then bench. Legality holds by construction.
        plan = [SLOT_MASKS[si] for si in
                sorted(range(TOTAL_STARTERS), key=lambda s: len(SLOT_MASKS[s]))] \
               + [None]*BENCH_COUNT                     # None = any position (bench)
        cur, spent, ncen, taken = [], 0, 0, set()
        for k, mask in enumerate(plan):
            slots_left = len(plan) - k
            avail = [i for i in live if i not in taken
                     and (mask is None or (b.eligs[i] & mask))
                     and not (b.isC[i] and ncen >= MAX_CENTERS)]
            if not avail:
                continue
            # reserve the cheapest way to fill every slot still open after this one
            others = sorted(b.price[j] for j in avail)[:slots_left-1] if slots_left > 1 else []
            cap = budget - spent - int(sum(others))
            fit = [i for i in avail if b.price[i] <= cap]
            pick = max(fit, key=lambda i: key[i]) if fit else min(avail, key=lambda i: b.price[i])
            cur.append(int(pick)); taken.add(int(pick))
            spent += int(b.price[pick]); ncen += int(b.isC[pick])
        return cur, spent

    cand = live[np.argsort(-b.val[live])][:n_cand]
    def improve(cur, spent):
        cur = list(cur); best = best_started_value(cur, b.val, b.eligs); inr = set(cur)
        for _ in range(passes):
            gain = False
            for pos in range(len(cur)):
                out = cur[pos]
                for i in cand:
                    if i in inr:
                        continue
                    ns = spent - b.price[out] + b.price[i]
                    if ns > budget:
                        continue
                    trial = cur[:pos] + cur[pos+1:] + [int(i)]
                    if sum(b.isC[j] for j in trial) > MAX_CENTERS:
                        continue
                    if not slots_fillable([b.eligs[j] for j in trial]):
                        continue
                    sc = best_started_value(trial, b.val, b.eligs)
                    if sc > best + 1e-9:
                        inr.discard(out); inr.add(int(i))
                        cur, spent, best, gain = trial, ns, sc, True
                        break
                if gain:
                    break
            if not gain:
                break
        return cur, spent, best

    results = []
    eff = b.val / np.maximum(b.price, 1)                 # value per dollar
    for key in (b.val, eff):
        cur, sp = greedy(key)
        if len(cur) == ROSTER_SIZE:
            results.append(improve(cur, sp))
    if not results:
        raise RuntimeError('no legal roster could be built')
    cur, spent, _ = max(results, key=lambda t: t[2])
    return finalize_roster(b, cur, spent)

def finalize_roster(b, idx, spent):
    """Attach SLOT / ROLE / PRICE_PAID and return the roster as a DataFrame."""
    idx = list(idx)
    r = b.df.iloc[idx].copy()
    r['PRICE_PAID'] = b.price[idx]
    r['SLOT'], r['ROLE'] = 'BENCH', 'BENCH'
    # match on positions WITHIN this roster, then map back to DataFrame rows
    local_val = np.array([b.val[i] for i in idx])
    local_el  = [b.eligs[i] for i in idx]
    filled = match_starters(range(len(idx)), local_val, local_el)
    pos_of = {local: r.index[local] for local in range(len(idx))}
    for si, local in filled.items():
        row = pos_of[local]
        r.at[row, 'SLOT'] = STARTER_SLOTS[si][0]
        r.at[row, 'ROLE'] = 'STARTER'
    # print order: starters in ESPN slot order (PG, SG, SF, PF, C, G, F, UTIL...), then bench
    slot_rank = {name: i for i, (name, _) in enumerate(STARTER_SLOTS)}
    r['_ord'] = [slot_rank.get(sl, 99) if ro == 'STARTER' else 99
                 for sl, ro in zip(r['SLOT'], r['ROLE'])]
    r = r.sort_values(['_ord', b.value_col], ascending=[True, False]).drop(columns='_ord')
    return r.reset_index(drop=True), int(spent)

def check_legal(r, spent, label=''):
    """Every roster this notebook prints must be legal under Cell 2's rules."""
    nc = sum(1 for e in r['POS_ELIGIBLE'] if 'C' in e)
    assert len(r) == ROSTER_SIZE,  f'{label}: {len(r)} players, need {ROSTER_SIZE}'
    assert spent <= BUDGET_PER_TEAM, f'{label}: spent ${spent} > ${BUDGET_PER_TEAM}'
    assert nc <= MAX_CENTERS,      f'{label}: {nc} centers > {MAX_CENTERS}'
    assert (r.ROLE == 'STARTER').sum() == TOTAL_STARTERS, f'{label}: starter slots unfilled'
    return True

print('✅ roster optimiser ready  ·  13 players · $200 · 10 ESPN slots · max 4 C')

---
## Cell 11 · The bundle search — letting the data pick the categories

Everything so far was setup. This cell answers the actual question.

For every category bundle of size 5, 6, 7, 8 and 10 — **628 of them** — it builds the optimal
$200 roster under that punt plan, simulates thousands of weekly matchups against a fixed
14-team field, and scores it on the only metric that matters: **P(win ≥ 6 of 10 categories)**.

### Method, and its one deliberate simplification

The field is **13 balanced managers** with ±12% valuation noise, which models competent but
not strategically-punting opponents. Each candidate bundle drafts from the **full player
pool**, so every bundle is judged against the *same* field and the same available talent.
That means a candidate may "share" a player with the field, which slightly flatters every
candidate — but it flatters them all equally, so the *ranking* is fair, and ranking is what
this cell is for. Cell 15's full auction removes the overlap and produces the honest
absolute numbers.

### What to read in the output

`P_win` is the headline. But watch the `ACTUALLY_LOCKS` column too, because it exposes
something important: **the target set is a valuation weighting, not a prediction of which
categories you win.** A roster built to maximise rebounds and double-doubles also happens to
pile up assists, because the players who do the first two do the third as well. The
categories you end up locking are an *emergent property* of the roster the plan buys — which
is precisely why this has to be simulated rather than reasoned about.

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# CELL 11 — BUNDLE SEARCH: all 628 category plans, ranked by P(win matchup)
# ═══════════════════════════════════════════════════════════════════
# Runtime note: ~2-4 minutes. Lower SEARCH_WEEKS or BUNDLE_SIZES to trade accuracy for speed.
SEARCH_WEEKS = 2500
BUNDLE_SIZES = (5, 6, 7, 8, 10)
SEARCH_NCAND, SEARCH_PASSES = 110, 4      # lighter optimiser: the search needs a fair ranking
LOCK_THRESHOLD = 0.80        # a category is "locked" at >=80% weekly win probability

# ── the fixed neutral field: 13 competent balanced managers ──
def build_field(seed, n=13, plans=None, exclude=None):
    """13 opponents. `plans` may give each a target set; default = balanced 10-cat."""
    rng = np.random.default_rng(seed)
    used = set(exclude) if exclude else set()
    out = []
    plans = plans or [tuple(CATS)]*n
    for i in range(n):
        pool = P.copy()
        pool['V'] = strategy_value(Z, list(plans[i]), pool['DUR_W']) \
                    * rng.uniform(0.88, 1.12, len(pool))     # imperfect managers
        r, s = build_roster(pool, 'V', 'MARKET_$', exclude=used,
                            n_cand=SEARCH_NCAND, passes=SEARCH_PASSES)
        check_legal(r, s, f'field team {i}')
        used |= set(r.PLAYER_ID)
        out.append(r)
    return out

FIELD = build_field(RNG_SEED)
FIELD_SIM = [sim_weeks(r, SEARCH_WEEKS, np.random.default_rng(RNG_SEED + 100 + i))
             for i, r in enumerate(FIELD)]

# ── draft position: candidates must NOT get first pick of an untouched board ──
# Earlier versions let every candidate draft from the full pool while the 13 field teams drafted
# sequentially around each other. That handed the candidate a large first-pick advantage, and it
# showed: the BALANCED plan scored 0.66-0.80 against a balanced field when by construction it
# should score ~0.50. Absolute numbers from that setup mean nothing.
# Fix: candidates draft from a board already depleted by the first MY_DRAFT_SLOT field teams, so
# they sit mid-pack like everyone else. The field is still built once and held fixed, so bundles
# stay directly comparable to one another.
MY_DRAFT_SLOT = 6                        # 7th of 14
DEPLETED = set().union(*[set(r.PLAYER_ID) for r in FIELD[:MY_DRAFT_SLOT]])

print(f'✅ field of {len(FIELD)} balanced opponents built and simulated')
print(f'   candidates draft {MY_DRAFT_SLOT+1}th of {NUM_TEAMS}: '
      f'{len(DEPLETED)} players already off the board')
print(f'\nLEAGUE WEEKLY BENCHMARK (median opponent team total per matchup):')
for c in CATS:
    med = float(np.median([np.median(f[c]) for f in FIELD_SIM]))
    print(f'   {LBL[c]:>4}  {med:9.3f}')

# ── evaluate one bundle ──
def evaluate_bundle(targets, field_sim=None, weeks=SEARCH_WEEKS, seed=0, exclude='depleted'):
    field_sim = field_sim if field_sim is not None else FIELD_SIM
    if isinstance(exclude, str) and exclude == 'depleted':
        exclude = DEPLETED
    pool = P.copy()
    pool['V'] = strategy_value(Z, list(targets), pool['DUR_W'])
    roster, spent = build_roster(pool, 'V', 'MARKET_$', exclude=exclude,
                                 n_cand=SEARCH_NCAND, passes=SEARCH_PASSES)
    mine = sim_weeks(roster, weeks, np.random.default_rng(RNG_SEED + 900 + seed))
    cw = {c: float(np.mean([cat_win_probs(mine, f)[c] for f in field_sim])) for c in CATS}
    mw = float(np.mean([matchup_win_prob(mine, f) for f in field_sim]))
    return roster, spent, cw, mw

t0 = time.time()
rows = []
for k in BUNDLE_SIZES:
    for tg in itertools.combinations(CATS, k):
        r, sp, cw, mw = evaluate_bundle(tg)
        locks = [LBL[c] for c in CATS if cw[c] >= LOCK_THRESHOLD]
        rows.append({'k': k, 'TARGETS': '+'.join(LBL[c] for c in tg), 'tset': tg,
                     'P_win': mw, 'n_locked': len(locks),
                     'ACTUALLY_LOCKS': '+'.join(locks),
                     'E_cats_won': sum(cw.values()), 'spend': sp,
                     **{LBL[c]: round(cw[c], 2) for c in CATS}})
df_bundles = pd.DataFrame(rows).sort_values('P_win', ascending=False).reset_index(drop=True)
print(f'\n✅ evaluated {len(df_bundles)} bundles in {time.time()-t0:.0f}s')

SHOW = ['k','TARGETS','P_win','n_locked','ACTUALLY_LOCKS','E_cats_won','spend']
print('\n' + '='*118)
print('  TOP 20 CATEGORY PLANS  —  ranked by P(win a matchup)')
print('='*118)
print(df_bundles.head(20)[SHOW].round(3).to_string(index=False))

print('\n' + '='*118)
print('  BEST PLAN AT EACH SIZE  —  this is the "how many categories should I target" answer')
print('='*118)
best_k = df_bundles.loc[df_bundles.groupby('k')['P_win'].idxmax()].sort_values('k')
print(best_k[SHOW].round(3).to_string(index=False))

bal = df_bundles[df_bundles.k == 10].iloc[0]
top = df_bundles.iloc[0]
print(f"\n  Balanced (all 10 cats)   P(win) = {bal.P_win:.3f}")
print(f"  Best concentrated plan   P(win) = {top.P_win:.3f}   <- {top.TARGETS}")
print(f"  EDGE FROM CONCENTRATING  {(top.P_win-bal.P_win)*100:+.1f} percentage points")
print()
print(f'  Sanity check: the balanced plan scores {bal.P_win:.3f}, and it SHOULD sit near 0.50 --')
print(f'  a balanced team against a balanced field is a coin flip. Drifting far above that means')
print(f'  candidates are getting too good a draft position; raise MY_DRAFT_SLOT if so.')
print(f'  Cell 15 remains the honest absolute number: there all 14 teams bid simultaneously.')

---
## Cell 12 · Robustness — the check that decides whether this is real

The result above was measured against a field of balanced managers. That raises the obvious
objection: **the edge comes from rivals ignoring the categories you attack, so what happens
when they don't?** A strategy that only works against passive opponents is not a strategy.

So this cell re-runs the leading plans against hostile fields:

- **3 rivals copy your plan** — the likely case if one or two managers in the league are sharp.
- **6 rivals copy your plan** — an implausibly hostile worst case.
- **3 rivals punt in the opposite direction** (a guard/shooting build) — tests whether being crowded out of the *other* categories hurts.

There is an asymmetry worth naming, because it is the reason this works at all:

**A punt is not contestable, but a target is.** If you concede FT% and 3PM, no rival can take
those away from you — you already gave them up. But the categories you *target* can be bid
away, which is why the contested columns are the honest test. A plan whose win probability
survives six rivals copying it is a plan that does not depend on the room being asleep.

Note also that in this cell **your team drafts first from the full pool**, which is the
best case. Cell 15's simultaneous auction is the realistic one.

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# CELL 12 — ROBUSTNESS: what happens when rivals contest your categories
# ═══════════════════════════════════════════════════════════════════
ROBUST_WEEKS = 5000

# the plans worth stress-testing: the search winners at each size, plus the
# guard/shooting build that intuition suggests first (so the comparison is explicit)
GUARD_PLAN = tuple(KEY[x] for x in ['PTS','PPM','AST','STL','3PM','FT%'])
CANDIDATES = {}
for _, row in best_k.iterrows():
    CANDIDATES[f"BEST-{row.k}  {row.TARGETS}"] = row.tset
CANDIDATES['GUARD-6  ' + '+'.join(LBL[c] for c in GUARD_PLAN)] = GUARD_PLAN

def contested_test(my_plan, rival_plan, n_rivals, seed, weeks=ROBUST_WEEKS):
    """My team drafts first from the full pool; then n_rivals copy rival_plan and the
    remaining opponents draft balanced, all from what is left."""
    rng = np.random.default_rng(seed)
    pool = P.copy(); pool['V'] = strategy_value(Z, list(my_plan), pool['DUR_W'])
    # same mid-pack draft position as the Cell 11 search (see MY_DRAFT_SLOT)
    mine, spent = build_roster(pool, 'V', 'MARKET_$', exclude=DEPLETED)
    check_legal(mine, spent, 'my team')
    plans = [tuple(rival_plan)]*n_rivals + [tuple(CATS)]*(13-n_rivals)
    rng.shuffle(plans)
    opp = build_field(seed, n=13, plans=plans, exclude=set(mine.PLAYER_ID))
    ms = sim_weeks(mine, weeks, np.random.default_rng(seed + 7000))
    osim = [sim_weeks(r, weeks, np.random.default_rng(seed + 8000 + i)) for i, r in enumerate(opp)]
    return float(np.mean([matchup_win_prob(ms, o) for o in osim])), spent

SCENARIOS = [('uncontested', 0, None), ('3 rivals copy you', 3, 'self'),
             ('6 rivals copy you', 6, 'self'), ('3 rivals punt guards', 3, GUARD_PLAN)]

t0 = time.time(); rob = []
for label, plan in CANDIDATES.items():
    row = {'MY PLAN': label}
    for sname, n, rival in SCENARIOS:
        rp = plan if rival == 'self' else (rival or plan)
        ps = [contested_test(plan, rp, n, RNG_SEED + s)[0] for s in (11, 29)]
        row[sname] = round(float(np.mean(ps)), 3)
    rob.append(row)
df_robust = pd.DataFrame(rob)
df_robust['WORST'] = df_robust[[s for s, _, _ in SCENARIOS]].min(axis=1)
df_robust = df_robust.sort_values('WORST', ascending=False)

print(f'robustness run in {time.time()-t0:.0f}s\n')
print('='*120)
print('  P(win matchup) UNDER HOSTILE FIELDS   —   WORST = the number to actually plan around')
print('='*120)
print(df_robust.to_string(index=False))

winner = df_robust.iloc[0]
print(f"\n  ⇒ Most robust plan: {winner['MY PLAN']}")
print(f"     holds P(win) >= {winner['WORST']:.3f} even in its worst scenario")

---
## Cell 13 · Your team, and the surplus board

The winning plan from Cell 12 is now locked in as `MY_PLAN` and used to build your roster and
your draft board.

**The surplus board is the single most useful table in this notebook.** It ranks players by

```
SURPLUS = MY_$ (your value, priced under your plan) − MARKET_$ (what the room pays)
```

A player with +$20 surplus is one you can win at market price and still profit. A player with
−$25 surplus is one the room will overpay for — **let him go, every time, no matter how good
he is.** In an auction the players you *skip* determine your team as much as the ones you buy,
because every dollar saved becomes a dollar of category edge somewhere else.

A note on what you will see in the roster: expect a **cheap, deep, unglamorous** team. The
plan buys production in categories the room does not price, which means very little of your
$200 goes to famous names. This is uncomfortable and it is the point — the roster is built to
win 6 categories every week, not to look good on draft night.

Watch for the `MAX_CENTERS = 4` constraint biting, and for the optimiser routing around it
with **PF-eligible rebounders**. That is the draft-day lesson: when the Center slots are gone,
the rebounds and double-doubles are still available from power forwards.

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# CELL 13 — YOUR TEAM + THE SURPLUS BOARD
# ═══════════════════════════════════════════════════════════════════
# Lock in the most robust plan found in Cell 12.
MY_PLAN  = CANDIDATES[df_robust.iloc[0]['MY PLAN']]
MY_PUNTS = [c for c in CATS if c not in MY_PLAN]

print('='*100)
print(f'  STRATEGY LOCKED  —  {YOUR_TEAM_NAME}')
print('='*100)
print(f"  TARGET ({len(MY_PLAN)}) : {' · '.join(LBL[c] for c in MY_PLAN)}")
print(f"  PUNT   ({len(MY_PUNTS)}) : {' · '.join(LBL[c] for c in MY_PUNTS)}")
print(f"  You need {CATS_TO_WIN} of 10 every week. How many categories are actually IN PLAY")
print(f"  -- the targets plus whatever free spillover comes with them -- is measured below.")

# ── your private valuation ──
P['VAL_MINE'] = strategy_value(Z, list(MY_PLAN), P['DUR_W'])

# The target roster, bought at MARKET prices -- this is the team to aim for.
my_team, my_spent = build_roster(P, 'VAL_MINE', 'MARKET_$')
check_legal(my_team, my_spent, 'my team')

# ── your prices: what each player is worth to YOUR plan ──
# MY_$ is the honest private value -- "if the whole league valued players the way my plan does,
# this is what he would cost." It is deliberately NOT rescaled to $200: the optimal roster is
# worth MORE than $200 in your own currency (that is exactly why the plan has an edge), and
# flattening that away would make you bid below market on everything and win nothing.
#
# The budget constraint belongs in the BIDDING policy, not in the valuation -- see Cell 14,
# where each bidder paces its $200 across 13 slots instead of paying full private value early.
P['MY_$']    = auction_prices(P['VAL_MINE'])
P['SURPLUS'] = P['MY_$'] - P['MARKET_$']
P['MAX_BID'] = np.maximum(P['MY_$'], MIN_BID)     # hard ceiling: never pay above your own value

_mine_total = int(P.loc[P['PLAYER_ID'].isin(my_team['PLAYER_ID']), 'MY_$'].sum())
print(f"\n  Your target roster costs ${my_spent} at MARKET prices but is worth ${_mine_total}")
print(f"  in your own currency -- a ${_mine_total - my_spent} edge on the same ${BUDGET_PER_TEAM}.")
print(f"  That gap IS the strategy; Cell 14 tests whether you can actually buy it at auction.")

# my_team was built before the calibrated prices existed, so attach them to its rows now.
my_team = my_team.merge(P[['PLAYER_ID', 'MY_$', 'SURPLUS', 'MAX_BID']],
                        on='PLAYER_ID', how='left')

ROSTER_COLS = ['SLOT','ROLE','PLAYER_NAME','TEAM','POS_STR','AGE','EXP_GP','RISK',
               'MARKET_$','MY_$','SURPLUS','P_PTS_M','P_REB_M','P_AST_M','P_STL_M',
               'P_BLK_M','P_DD_M','P_FG3M_M']
print('\n' + '='*140)
print(f'  🏆  {YOUR_TEAM_NAME.upper()} — OPTIMAL ROSTER   (spend ${my_spent} / ${BUDGET_PER_TEAM})')
print('='*140)
print(my_team[ROSTER_COLS].round(2).to_string(index=False))

st = my_team[my_team.ROLE == 'STARTER']
n_cen = sum(1 for e in my_team['POS_ELIGIBLE'] if 'C' in e)
print(f"\n  starters ${int(st['MARKET_$'].sum())} · bench "
      f"${int(my_team.loc[my_team.ROLE=='BENCH','MARKET_$'].sum())}"
      f" · centers {n_cen}/{MAX_CENTERS}"
      + ('  <- Center cap BINDING: extra rebounding must come from PF-eligible players'
         if n_cen >= MAX_CENTERS else ''))

# Per-category weekly projection vs the league benchmark.
# Both sides must be simulated over the SAME number of weeks -- the comparison is week-by-week,
# so a 20k-week array cannot be compared against the 2.5k-week field used for the search.
FINAL_WEEKS = 12000
FIELD_SIM_HI = [sim_weeks(r, FINAL_WEEKS, np.random.default_rng(RNG_SEED + 400 + i))
                for i, r in enumerate(FIELD)]
mine_sim = sim_weeks(my_team, FINAL_WEEKS, np.random.default_rng(RNG_SEED + 1))
print(f"\n  ── YOUR PROJECTED WEEKLY TOTALS vs THE LEAGUE ──")
print(f"  {'CAT':>5} {'YOU':>10} {'LEAGUE MED':>12} {'P(win)':>8}  {'PLAN':<7} VERDICT")
cat_pw = {}
for c in CATS:
    opp_med = float(np.median([np.median(f[c]) for f in FIELD_SIM_HI]))
    pw = float(np.mean([cat_win_probs(mine_sim, f)[c] for f in FIELD_SIM_HI]))
    cat_pw[c] = pw
    tag = 'TARGET' if c in MY_PLAN else 'punt'
    if   pw >= LOCK_THRESHOLD: verdict = 'LOCKED  🔒'
    elif pw >= 0.60:           verdict = 'leaning ↗'
    elif pw >= 0.40:           verdict = 'live coin-flip' + ('   <- FREE BUFFER' if tag == 'punt' else '')
    else:                      verdict = 'conceded ✗'
    print(f"  {LBL[c]:>5} {np.median(mine_sim[c]):10.3f} {opp_med:12.3f} {pw:8.2f}  {tag:<7} {verdict}")

locked  = [LBL[c] for c in CATS if cat_pw[c] >= LOCK_THRESHOLD]
leaning = [LBL[c] for c in CATS if 0.60 <= cat_pw[c] < LOCK_THRESHOLD]
live    = [LBL[c] for c in CATS if 0.40 <= cat_pw[c] < 0.60]
in_play = len(locked) + len(leaning) + len(live)
print(f"\n  LOCKED ({len(locked)}): {' '.join(locked) or '-'}"
      f"   |   LEANING ({len(leaning)}): {' '.join(leaning) or '-'}"
      f"   |   LIVE ({len(live)}): {' '.join(live) or '-'}")
print(f"  ⇒ {in_play} categories in play; you need {CATS_TO_WIN}. "
      f"Expected won per matchup: {sum(cat_pw.values()):.2f}")

# Honesty check: a target you are not actually winning is a wasted target slot.
weak = [LBL[c] for c in MY_PLAN if cat_pw[c] < 0.60]
if weak:
    print(f"\n  ⚠ TARGETS THAT ARE NOT PAYING: {', '.join(weak)}")
    print(f"    Won under 60% of weeks despite being in the plan. They stay because they ride")
    print(f"    along with players you buy anyway (PTS and PPM correlate ~0.90) -- not because")
    print(f"    they earn their own dollars. Do NOT bid extra for a specialist in them.")

# the surplus board: who to buy, and who to let go
board = P[P['MARKET_$'] > 0].copy()
BOARD_COLS = ['PLAYER_NAME','TEAM','POS_STR','AGE','EXP_GP','RISK','MARKET_$','MY_$','SURPLUS',
              'P_PTS_M','P_REB_M','P_AST_M','P_STL_M','P_BLK_M','P_DD_M']
print('\n' + '='*130)
print('  💰  BUY THESE — biggest SURPLUS (worth far more to your plan than to the room)')
print('='*130)
print(board.nlargest(25, 'SURPLUS')[BOARD_COLS].round(2).to_string(index=False))
print('\n' + '='*130)
print('  🚫  LET THESE GO — the room will overpay; every dollar you skip here buys edge elsewhere')
print('='*130)
print(board.nsmallest(15, 'SURPLUS')[BOARD_COLS].round(2).to_string(index=False))

---
## Cell 14 · The full auction — who gets drafted by whom, for what price

This is the simulation of the actual draft night: **14 budgeted managers bidding against each
other simultaneously**, rather than 14 teams taking turns off a ranked list.

### How it works

**Nomination** rotates around the room. 70% of the time a manager nominates the player he most
wants (highest surplus on *his* board); 30% of the time he throws out an expensive player he
does *not* want, to drain rivals' budgets. Both are real auction behaviours and the mix
matters, because budget-draining is what creates the late-draft bargains.

**Bidding.** Every manager who can legally use the player computes a private ceiling from
his own valuation, adjusted for:

- **Budget pacing** — his remaining budget divided by what his *own remaining wish list* would cost at his private values. Below 1 he discounts every bid; above 1 (money left, talent thinning) he bids over private value, which is where auction inflation actually comes from. Pacing per-manager rather than league-wide matters because managers do not face the same scarcity — they do not want the same players.
- **Slot scarcity** — a small premium for a still-unfilled tight slot (PG/SG/SF/PF/C).
- **The $1 reserve** — you can never bid so much that you cannot afford $1 for each remaining empty spot. This is the constraint that actually shapes endgame behaviour.

**Clearing price = second-highest ceiling + $1.** This is the crucial mechanic and the reason
the whole strategy pays: in an English auction you pay one dollar more than the *second*-most
interested manager, not what the player is worth to you. So when you are the only manager who
values double-doubles, you get them near the minimum — and when you and three rivals all want
the same scorer, you pay far above his worth. **Bargains are a function of how many other
people want the player, not of how good he is.**

A note on why pacing is the mechanic that matters: paying full private value early is how you
end a draft with four stars and nine empty roster spots. In this format that is fatal, because
**all ten starters score every week** — a lineup with six near-worthless starters loses
categories it had no business losing. An earlier version of this notebook let bidders pay their
full private value, and the punt team bought four studs for $174 and finished last in the
simulated season. The valuation was right; the spending policy was wrong.

### How to read the output

The per-team roster tables are the answer to "which players get drafted by each team for what
price." `PAID vs MARKET` shows who overpaid. And the numbers here are more conservative than
Cell 12's, because your team no longer drafts first from an untouched pool — it fights for
every player like everyone else. **These are the honest numbers.**

### How realistic are these prices?

The cell prints a **price realism check**, because a simulated auction whose prices do not behave
like real ones is worse than no simulation at all — and the failure is silent. Three earlier
versions of the bidding model failed it, each in a different way, and the comments in the code name
them: full-private-value bidding (four stars for $174, last place), uniform budget pacing (Jokić for
$32 against a $91 board), and a static spending envelope (every ceiling pinned to $1 and **104 of
182 players sold at the minimum**, Brandon Miller at $1 against a $10 board price).

What fixed it was anchoring every ceiling to the market board and paying a premium only for private
surplus — which is how managers actually bid, since everybody can see ESPN's numbers. Prices now
correlate with the board at about **0.97**, and the top of the draft clears within a dollar or two
of market.

Two things to read honestly in that check:

**A large $1 tail is correct, not suspicious.** ESPN prices **53% of the 182 players who get
drafted at $2 or less**, so an auction where ~57% of sales land at the minimum is reproducing the
board's shape. (An earlier version of this check compared against an invented "30–45% is normal"
and cried wolf at a tail the data fully justified.)

**The mid-tier is mildly too cheap.** Roughly 15–20 players the board prices at $6+ still clear at
$1, because this simulation has exactly as many roster spots as drafted players — so the last names
necessarily face a single buyer. Real leagues hold undrafted free agents in reserve, which keeps the
mid-tier contested longer. The bias runs one way: **late-draft bargains here are a little better
than you will actually get.** Nothing about the category strategy depends on it; that comes from
Cell 12.

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# CELL 14 — FULL AUCTION SIMULATION (14 bidders, real prices)
# ═══════════════════════════════════════════════════════════════════
# How much of your private surplus you must pay to win a player you value above the market.
# Exported to league_meta.json in Cell 17 so the Streamlit app bids on exactly this number
# rather than keeping its own copy.
SURPLUS_CAPTURE = 0.60

class Bidder:
    """One manager: a punt plan, a private $ valuation, a budget and 13 slots.

    Budget discipline is an **adaptive per-slot reserve**: before each bid the manager sets aside
    a floor price for every slot he still has to fill, where the floor is a fraction of his own
    current budget-per-remaining-slot. Because it is recomputed from the live budget it can never
    strand him, and it can never collapse.

    That recomputation is the point, and it took three attempts to get right:
      * ceiling = full private value, reserve $1/slot -> bought 4 stars for $174 and finished
        LAST: six of its ten starters were worthless, and all ten starters score every week.
      * ceiling = value x (budget / cost of my top 13) -> every manager discounted to ~31% of
        value at once, so nothing cleared near market and Jokic sold for $32 against $91.
      * reserve = a STATIC envelope of the planned roster's market prices -> once a few purchases
        came in above plan, the fixed reserve exceeded the remaining budget, every manager's
        max_bid pinned to $1, and 104 of 182 players cleared at the minimum. Brandon Miller
        ($10 market, 21 ppg) went for $1 with all fourteen ceilings at $1, which no real auction
        does. An adaptive reserve cannot get into that state.
    """
    def __init__(self, name, targets, dollars, market, is_me=False, aggression=1.0,
                 surplus_capture=SURPLUS_CAPTURE):
        self.name, self.targets, self.is_me = name, tuple(targets), is_me
        self.val = np.asarray(dollars, float)
        self.market = np.asarray(market, float)      # the board everyone can see
        self.aggr = aggression
        self.capture = surplus_capture
        self.budget = BUDGET_PER_TEAM
        self.roster, self.prices, self.ncen = [], [], 0
        self.slots_open = [SLOT_MASKS[si] for si in
                           sorted(range(TOTAL_STARTERS), key=lambda s: len(SLOT_MASKS[s]))] \
                          + [None]*BENCH_COUNT
    @property
    def spots(self):   return ROSTER_SIZE - len(self.roster)
    @property
    def done(self):    return self.spots == 0
    @property
    def max_bid(self):
        """Budget, keeping $1 for each slot still to fill. Nothing more elaborate is needed:
        because every ceiling is anchored to the market board (see wtp), prices land near market
        and a $200 budget buys about 1/14th of a $2,800 board on its own. It is the anchoring that
        allocates the budget sensibly, not the reserve."""
        return int(max(MIN_BID, self.budget - (self.spots - 1) * MIN_BID))

    def slot_for(self, elig):
        typed = [s for s in self.slots_open if s is not None and (elig & s)]
        if typed:  return min(typed, key=len)
        return 'BENCH' if None in self.slots_open else None

    def can_take(self, i, eligs, isC):
        if self.done or (isC[i] and self.ncen >= MAX_CENTERS): return False
        return self.slot_for(eligs[i]) is not None

    def wtp(self, i, eligs, isC):
        """Willingness to pay = the market price, plus a share of this manager's private surplus.

            ceiling = market_i + (my_value_i - market_i) x capture

        This is the mechanic that makes the simulated prices behave like real ones, and it is how
        managers actually bid: everyone can see ESPN's board, so nobody lets a $19 player go for
        $1, but a manager pays over the board only for players his build actually wants.

        Anchoring also fixes budget allocation for free. If prices sit near market, and the market
        board's top 182 sums to about one league's worth of money, then $200 naturally buys roughly
        a 1/14th share -- no pacing heuristic required. Bidding full private value instead sent
        every manager broke by the middle of the draft, after which 59% of the pool cleared at the
        $1 minimum and genuinely useful players (Desmond Bane, $19 market) went for $5.
        """
        if not self.can_take(i, eligs, isC):
            return 0
        mkt = self.market[i]
        v = (mkt + (self.val[i] - mkt) * self.capture) * self.aggr
        s = self.slot_for(eligs[i])
        if s != 'BENCH' and len(s) <= 1:
            v *= 1.08                                   # scarce typed slot still unfilled
        if v < MIN_BID:
            return 0          # no $1 floor: free bodies would burn the scarce Center allowance
        return int(max(0, min(v, self.max_bid)))

    def take(self, i, price, eligs, isC):
        s = self.slot_for(eligs[i])
        self.slots_open.remove(s if s != 'BENCH' else None)
        self.roster.append(int(i)); self.prices.append(int(price))
        self.budget -= int(price); self.ncen += int(isC[i])

def run_auction(df, bidders, price_col='MARKET_$', rng=None):
    """Simultaneous English auction. Players clear at second-highest WTP + $1."""
    rng = rng or np.random.default_rng(RNG_SEED)
    eligs = list(df['POS_ELIGIBLE']); isC = np.array(['C' in e for e in eligs])
    mkt = df[price_col].clip(lower=MIN_BID).to_numpy(int)
    names = df['PLAYER_NAME'].to_numpy(); pids = df['PLAYER_ID'].to_numpy()
    avail = np.ones(len(df), bool)       # still on the board
    passed = np.zeros(len(df), bool)     # nominated, nobody bid -> skip in nominations, but they
                                         # stay on the board for the fill phase
    log = []; turn = 0; guard = 0

    while any(not b.done for b in bidders) and guard < 5000:
        guard += 1
        active = [b for b in bidders if not b.done]
        pool = np.flatnonzero(avail & ~passed)
        if len(pool) == 0:
            break                        # nothing left anyone will bid real money on

        nom = active[turn % len(active)]; turn += 1
        can = [i for i in pool if nom.can_take(i, eligs, isC)]
        if not can:
            can = [i for i in pool if not isC[i]] or list(pool)
        # Nomination mix. The random third matters more than it looks: when nominations always
        # chase the most valuable player, the whole expensive top of the board sells first, every
        # budget is drained by the midpoint, and the $6-20 mid-tier then clears at $1 because
        # nobody can afford anything. Real auctions sprinkle mid-priced players throughout, which
        # is what keeps their prices honest.
        r = rng.random()
        if r < 0.50:                 # the player you most want at a fair price
            pick = max(can, key=lambda i: nom.val[i] - mkt[i])
        elif r < 0.70:               # throw out a star to drain rival budgets
            pick = max(can, key=lambda i: mkt[i])
        else:                        # arbitrary name off the board
            pick = int(rng.choice(can))

        bids = sorted(((b.wtp(pick, eligs, isC), b) for b in bidders
                       if b.can_take(pick, eligs, isC)), key=lambda t: -t[0])
        if not bids or bids[0][0] < MIN_BID:
            # Nobody will pay above the minimum. Do NOT remove him from the board -- he is exactly
            # the kind of player the fill phase needs. Just stop nominating him, or the loop spins.
            passed[pick] = True
            continue
        win = bids[0][1]
        second = bids[1][0] if len(bids) > 1 else 0
        # An English auction clears at one dollar above the SECOND-highest ceiling: you pay what
        # it took to outlast the next-most-interested manager, not what the player is worth to
        # you. This is why bargains exist wherever you are the only bidder who wants someone.
        price = max(MIN_BID, min(bids[0][0], second + 1))
        win.take(pick, price, eligs, isC)
        avail[pick] = False
        log.append({'PLAYER_ID': pids[pick], 'PLAYER_NAME': names[pick], 'TEAM': win.name,
                    'PAID': price, 'MARKET': int(mkt[pick]), 'N_BIDDERS': len(bids),
                    'RUNNER_UP': bids[1][1].name if len(bids) > 1 else '-',
                    'CEILING': bids[0][0]})
    # ── fill phase: the end of a real auction, where leftover slots go for $1 ──
    # Runs only once nobody will bid above the minimum any more, so it can no longer cannibalise
    # a scarce positional allowance the way an early $1 centre did.
    for b in bidders:
        while b.spots > 0:
            cands = [i for i in np.flatnonzero(avail) if b.can_take(i, eligs, isC)]
            if not cands:
                break
            pick = max(cands, key=lambda i: b.val[i])      # best of the leftovers, not arbitrary
            b.take(pick, MIN_BID, eligs, isC)
            avail[pick] = False
            log.append({'PLAYER_ID': pids[pick], 'PLAYER_NAME': names[pick], 'TEAM': b.name,
                        'PAID': MIN_BID, 'MARKET': int(mkt[pick]), 'N_BIDDERS': 1,
                        'RUNNER_UP': '- (fill)', 'CEILING': MIN_BID})

    return pd.DataFrame(log)

# ── set up the room: you + 13 rivals. 3 rivals punt, the rest play balanced. ──
opp_names = [n for n in TEAM_NAMES if n != YOUR_TEAM_NAME]
rng = np.random.default_rng(RNG_SEED)
RIVAL_PUNTS = [tuple(KEY[x] for x in ['PTS','PPM','3PM','AST','FT%','STL']),   # guard punt
               tuple(KEY[x] for x in ['REB','BLK','DD','FG%','PTS','AST']),    # big punt
               tuple(KEY[x] for x in ['AST','STL','3PM','PTS','FT%','PPM'])]   # another guard punt

# Every bidder gets its private valuation in the same currency (auction_prices), and paces its
# own $200 during the auction -- so punting rivals face the same budget discipline we do.
MKT = P['MARKET_$'].to_numpy(float)
bidders = [Bidder(f'>>> {YOUR_TEAM_NAME}', MY_PLAN, P['MY_$'].to_numpy(), MKT, is_me=True)]
for j, nm in enumerate(opp_names):
    if j < len(RIVAL_PUNTS):
        plan = RIVAL_PUNTS[j]
        dollars = auction_prices(strategy_value(Z, list(plan), P['DUR_W']))
    else:
        plan = tuple(CATS)
        dollars = P['MARKET_$'].to_numpy()
    bidders.append(Bidder(nm, plan, dollars * rng.uniform(0.92, 1.08), MKT,
                          aggression=float(rng.uniform(0.92, 1.10))))

df_sales = run_auction(P, bidders, rng=rng)
short = [b.name for b in bidders if b.spots > 0]
assert not short, f'auction left rosters incomplete: {short}'
assert len(df_sales) == TOTAL_DRAFTED, f'{len(df_sales)} sold, expected {TOTAL_DRAFTED}'
print(f'✅ auction complete: {len(df_sales)} players sold, '
      f"${df_sales['PAID'].sum():,} of ${TOTAL_BUDGET:,} spent — all 14 rosters full\n")

# ── rebuild every team's roster from the auction result ──
auction_teams, auction_spend = {}, {}
for b in bidders:
    t = P.iloc[b.roster].copy()
    t['PAID'] = b.prices
    t, _ = finalize_roster(Board(t.assign(_V=strategy_value(Z, list(b.targets), t['DUR_W'])),
                                 '_V', 'PAID'), range(len(t)), sum(b.prices))
    auction_teams[b.name] = t
    auction_spend[b.name] = sum(b.prices)

# ── price realism check ──────────────────────────────────────────────────────
# A simulated auction is only useful if its prices look like an auction's, and the failure mode is
# silent. The benchmark has to be the MARKET BOARD's own shape, not a round number: ESPN prices
# most rosterable players at the minimum, so a large $1 tail is correct rather than suspicious.
# (An earlier version of this check compared against an invented "30-45% is normal" and cried wolf
# at a 62% tail that the board fully justified.) What actually indicates a broken auction is
# MID-TIER players -- ones the board prices at $6+ -- clearing at the minimum, which is what
# happens when every manager is broke at the same time.
board = P['MARKET_$'].nlargest(TOTAL_DRAFTED)
board_min_share = float((board <= 2).mean())
n1 = int((df_sales.PAID == MIN_BID).sum())
squeezed = df_sales[(df_sales.PAID == MIN_BID) & (df_sales.MARKET >= 6)]
corr = df_sales[['PAID','MARKET']].corr().iloc[0, 1]
print('── price realism ──')
print(f'   sales at the ${MIN_BID} minimum   : {n1}/{len(df_sales)} ({n1/len(df_sales)*100:.0f}%)'
      f'   vs {board_min_share*100:.0f}% of the board priced <= $2 — these should be close')
print(f'   $6+ players sold for ${MIN_BID} : {len(squeezed)}'
      f'   — mid-tier squeeze; expected to be smallish, see the note below')
print(f'   corr(paid, market)        : {corr:.3f}   — prices should track the board')
if corr < 0.85:
    print('   ⚠ prices have come loose from the board — check Bidder.wtp and max_bid.')
else:
    print(f'   ✓ prices track the board (corr {corr:.2f}) and the $1 tail matches its shape.')
print(f'\n   KNOWN LIMITATION — the {len(squeezed)} mid-tier players ($6+ on the board) that sold'
      f' for ${MIN_BID}:')
print('   there are exactly as many roster spots as drafted players, so the last names on the')
print('   board necessarily have one interested buyer and go for the minimum. Real leagues keep')
print('   undrafted free agents in reserve, so their mid-tier stays contested a while longer.')
print('   DIRECTION OF BIAS: late-draft bargains here are slightly better than you will get, so')
print('   read the auction roster as a mildly optimistic version of draft night. It does not')
print('   affect which categories to target — that comes from the Cell 12 search, not from here.')
print()

print('='*118)
print(f'  🏛  AUCTION RESULT — EVERY TEAM, EVERY PLAYER, EVERY PRICE')
print('='*118)
for name, t in auction_teams.items():
    nc = sum(1 for e in t['POS_ELIGIBLE'] if 'C' in e)
    over = int((t['PAID'] - t['MARKET_$']).sum())
    print(f"\n  {name}   —   ${auction_spend[name]}/{BUDGET_PER_TEAM} spent · "
          f"{len(t)} players · {nc} C · paid {over:+d} vs market")
    print(t.sort_values('PAID', ascending=False)[
        ['SLOT','ROLE','PLAYER_NAME','TEAM','POS_STR','PAID','MARKET_$','EXP_GP','RISK',
         'P_PTS_M','P_REB_M','P_AST_M','P_STL_M','P_BLK_M','P_DD_M']
    ].round(2).to_string(index=False))

print('\n' + '='*118)
print('  💸  BIGGEST OVERPAYS IN THE ROOM  (paid − market)')
print('='*118)
df_sales['OVER'] = df_sales['PAID'] - df_sales['MARKET']
print(df_sales.nlargest(12, 'OVER')[['PLAYER_NAME','TEAM','PAID','MARKET','OVER',
                                     'N_BIDDERS','RUNNER_UP','CEILING']].to_string(index=False))
print('\n  🎯  YOUR BARGAINS (players you won below market)')
mine_sales = df_sales[df_sales.TEAM.str.contains(YOUR_TEAM_NAME)].nsmallest(12, 'OVER')
print(mine_sales[['PLAYER_NAME','PAID','MARKET','OVER','N_BIDDERS','RUNNER_UP']].to_string(index=False))

---
## Cell 15 · Season Monte Carlo — the record this actually produces

Per-matchup win probability is the right optimisation target, but it is not what you look at on
the last day of the regular season. This cell simulates **complete 18-matchup seasons** week by
week for all 14 teams: every week each team's category totals are drawn fresh, teams are
paired, and the 6-of-10 rule settles each matchup.

**It runs on the rosters from the auction in Cell 14, not on the idealised roster from Cell
13 — and that distinction is the difference between an honest projection and a flattering
one.** In Cell 13 your team was built from the untouched player pool while the field divided
up what was left, which hands you an advantage no real draft would. The auction makes all 14
teams bid against each other for the same players, so nobody gets a free run at the board.

Both numbers are reported so the gap is visible: the **idealised** roster is the target you are
aiming at, and the **auction** roster is what a realistic draft night delivered. If the auction
result is much weaker, that is not a modelling failure — it is the cost of live competition,
and it is the number to plan around.

Two outputs matter:

**The category rate** (categories won per matchup) shows *how* you win. A concentrated build
should cluster tightly at 6, 7 and 8 categories — **not** spread broadly. A broad spread means
you are still relying on variance, which is the thing this whole strategy exists to remove.

**Matchup record and playoff odds** are the real bottom line. With `PLAYOFF_TEAMS = 4` of 14
you need roughly a top-29% finish, and the 10th–90th percentile range matters more than the
average: a build averaging 12-6 that ranges 7-11 to 16-2 is a worse bet than one averaging
11-7 that never drops below 9-9.

### The assumption this projection rests on

The 13 rivals price players at **ESPN's published auction values**, which are computed for ESPN's
default 9-category scoring — a scoring set with **no DD and no PPM**. This league scores both. So
in this simulation the rebounding centres who pile up REB and double-doubles are genuinely cheap
(Zubac at $4, Kel'el Ware at $3), and a plan built on those categories buys a dominant roster for
very little. That is why the projected record here is strong.

**If the managers in your league have noticed that DD is scored, those players will not be that
cheap**, and this number comes down. Cell 12's contested columns are the hedge: they re-measure the
plan with 3 and 6 rivals attacking the same categories, and it is those worst-case figures — not
this projection — that you should plan around. Treat this cell as "what happens if the room drafts
off ESPN's board," which is the common case but not a guarantee.

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# CELL 15 — SEASON MONTE CARLO: 18 matchups, record, playoff odds
# ═══════════════════════════════════════════════════════════════════
N_SEASONS = 400

def simulate_season(rosters, n_seasons=N_SEASONS, seed=0):
    """Full 18-matchup seasons. Each week every team draws fresh weekly totals, teams are
    paired at random, and each matchup is settled on the 6-of-10 rule (ties count half)."""
    rng = np.random.default_rng(seed)
    n = len(rosters)
    weeks = REGULAR_SEASON_MATCHUPS * n_seasons
    sims = [sim_weeks(r, weeks, np.random.default_rng(seed + 31*i)) for i, r in enumerate(rosters)]
    wins = np.zeros((n_seasons, n)); catw = np.zeros((n_seasons, n))
    for s in range(n_seasons):
        for w in range(REGULAR_SEASON_MATCHUPS):
            idx = s*REGULAR_SEASON_MATCHUPS + w
            order = rng.permutation(n)
            for a, b in zip(order[0::2], order[1::2]):
                ca = 0.0
                for c in CATS:
                    va, vb = sims[a][c][idx], sims[b][c][idx]
                    ca += 1.0 if va > vb else (0.5 if va == vb else 0.0)
                catw[s, a] += ca; catw[s, b] += (len(CATS) - ca)
                if   ca > len(CATS)/2: wins[s, a] += 1
                elif ca < len(CATS)/2: wins[s, b] += 1
                else:                  wins[s, a] += 0.5; wins[s, b] += 0.5
    return wins, catw

def season_table(rosters, names, seed):
    wins, catw = simulate_season(rosters, seed=seed)
    played = REGULAR_SEASON_MATCHUPS
    rank = (-wins).argsort(axis=1).argsort(axis=1)
    return pd.DataFrame({
        'TEAM': names,
        'W': wins.mean(axis=0).round(1),
        'L': (played - wins.mean(axis=0)).round(1),
        'W_p10': np.percentile(wins, 10, axis=0).round(1),
        'W_p90': np.percentile(wins, 90, axis=0).round(1),
        'CATS/wk': (catw.mean(axis=0)/played).round(2),
        'PLAYOFF%': ((rank < PLAYOFF_TEAMS).mean(axis=0)*100).round(1),
        'BEST_RECORD%': ((rank == 0).mean(axis=0)*100).round(1),
    }).sort_values('W', ascending=False)

# ── THE HONEST PROJECTION: the 14 rosters that came out of the live auction ──
auction_names = list(auction_teams.keys())
summary = season_table([auction_teams[n] for n in auction_names], auction_names, RNG_SEED)

print('='*106)
print(f'  📊  PROJECTED {REGULAR_SEASON_MATCHUPS}-MATCHUP SEASON — from the Cell 14 AUCTION rosters')
print(f'      {LEAGUE_NAME} · top {PLAYOFF_TEAMS} of {NUM_TEAMS} make the playoffs')
print('='*106)
print(summary.to_string(index=False))

me = summary[summary.TEAM.str.contains(YOUR_TEAM_NAME)].iloc[0]
played = REGULAR_SEASON_MATCHUPS
print(f"\n  {YOUR_TEAM_NAME} (auction roster): {me.W:.1f}-{me.L:.1f}"
      f"   ·   playoffs {me['PLAYOFF%']:.1f}%   ·   best record {me['BEST_RECORD%']:.1f}%")
print(f"  10th-90th percentile: {me.W_p10:.0f}-{played-me.W_p10:.0f} to {me.W_p90:.0f}-{played-me.W_p90:.0f}")
print(f"  Categories won per matchup: {me['CATS/wk']:.2f} of {len(CATS)}  (need {CATS_TO_WIN})")

# ── for comparison only: the idealised Cell 13 roster against the same field ──
ideal_names = [f'>>> {YOUR_TEAM_NAME} (ideal)'] + [n for n in auction_names
                                                   if YOUR_TEAM_NAME not in n]
ideal = season_table([my_team] + [auction_teams[n] for n in auction_names
                                  if YOUR_TEAM_NAME not in n], ideal_names, RNG_SEED + 5)
mi = ideal[ideal.TEAM.str.contains('ideal')].iloc[0]
print(f"\n  For comparison — the IDEALISED Cell 13 roster vs the same field:")
print(f"    {mi.W:.1f}-{mi.L:.1f} · playoffs {mi['PLAYOFF%']:.1f}% · {mi['CATS/wk']:.2f} cats/matchup")
print(f"    Gap to the auction result: {mi.W - me.W:+.1f} wins. That gap is the price of")
print(f"    competing for players instead of getting first pick of the board.")

# ── shape of the win: how many categories do you take per matchup? ──
dist = np.zeros(len(CATS)+1)
for f in FIELD_SIM_HI:
    w = cats_won_dist(mine_sim, f)
    for k in range(len(CATS)+1):
        dist[k] += ((w >= k-0.25) & (w <= k+0.25)).sum()
dist = dist/dist.sum()
print(f'\n  ── DISTRIBUTION OF CATEGORIES WON PER MATCHUP (idealised roster) ──')
for k in range(len(CATS)+1):
    print(f"   {k:2d} cats  {dist[k]*100:5.1f}%  {'█'*int(round(dist[k]*90))}"
          f"{'  <- WIN' if k >= CATS_TO_WIN else ''}")
print(f'\n  P(win matchup) = {dist[CATS_TO_WIN:].sum()*100:.1f}%')

---
## Cell 16 · Draft-night cheat sheet

The auction above is one simulated draft night; the real one will diverge from it within ten
nominations. So this last cell produces the thing you actually use at the table: **hard
numbers and decision rules that survive the draft not going to plan.**

Three rules worth internalising before you sit down:

**1. `MAX_BID` is your own valuation, and it is a hard ceiling.** Paying $1 over what a player
is worth to your plan converts your edge into someone else's. The discipline to stop bidding
is the strategy; everything upstream in this notebook is just arithmetic that tells you where
to stop.

**2. Punts are permanent, targets are negotiable.** If rivals bid your top rebounder past your
ceiling, you do not abandon rebounds — you buy the next rebounder. But never "rescue" a punt
category because a bargain appears there: a cheap elite three-point shooter is still a wasted
dollar when you have conceded 3PM, because he wins you a category you already gave away.

**3. Spend on availability, not on names.** With no IR slot, no lineup protection and weekly
locks, an injured starter posts zeros across your whole target set for a full matchup. A
category lock is only a lock while the player is on the floor — which is why the durability
exponent in Cell 2 is convex and why the board favours the healthy.

And once the season starts: you get **7 acquisitions per matchup** and **$100 FAB**. Stream
against your own plan — pick up whoever adds to your target categories for the week and ignore
production in your punts entirely, no matter how good it looks.

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# CELL 16 — DRAFT-NIGHT CHEAT SHEET + EXCEL EXPORT
# ═══════════════════════════════════════════════════════════════════
print('='*112)
print(f'  🎯  DRAFT NIGHT CHEAT SHEET  —  {YOUR_TEAM_NAME}  ·  {DRAFT_DATE:%b %d, %Y @ %H:%M}')
print('='*112)
print(f"  TARGET : {' · '.join(LBL[c] for c in MY_PLAN)}")
print(f"  PUNT   : {' · '.join(LBL[c] for c in MY_PUNTS)}   <- never spend here, not even on a bargain")
print(f"  Budget ${BUDGET_PER_TEAM} · {ROSTER_SIZE} spots · max {MAX_CENTERS} C · "
      f"only the {TOTAL_STARTERS} starters score")

# tier the board by your own valuation, with the hard ceiling for each player
cheat = P[P['MY_$'] > 0].sort_values('MY_$', ascending=False).copy()
cheat['TIER'] = pd.cut(cheat['MY_$'], [-1, 5, 12, 25, 45, 999],
                       labels=['$1-5 filler','$6-12 value','$13-25 core','$26-45 anchor','$46+ elite'])
# ESPN_$_LIST is what ESPN's UI shows; MARKET_$ is that inflated to a 14-team full-spend
# auction. Bid against MAX_BID, but knowing ESPN's number tells you what the room is anchored to.
CHEAT_COLS = ['PLAYER_NAME','TEAM','POS_STR','AGE','EXP_GP','RISK','MAX_BID','MARKET_$',
              'ESPN_$_LIST','SURPLUS']
for tier in ['$46+ elite','$26-45 anchor','$13-25 core','$6-12 value']:
    grp = cheat[cheat.TIER == tier]
    if grp.empty: continue
    print(f"\n  ── {tier}  ({len(grp)} players) ──")
    print(grp.head(14)[CHEAT_COLS].to_string(index=False))

print('\n  ── $1-3 END-GAME FILLERS with real target-category production ──')
fill = P[(P['MY_$'] <= 3)].nlargest(15, 'VAL_MINE')
print(fill[['PLAYER_NAME','TEAM','POS_STR','EXP_GP','MY_$','P_REB_M','P_AST_M','P_STL_M',
            'P_BLK_M','P_DD_M','P_PTS_M']].round(2).to_string(index=False))

print(f"\n  ── PF-ELIGIBLE, NON-CENTER rebounding (your route around the {MAX_CENTERS}-Center cap) ──")
pf_reb = P[[('PF' in e and 'C' not in e) for e in P['POS_ELIGIBLE']]].nlargest(12, 'P_REB_M')
print(pf_reb[['PLAYER_NAME','TEAM','POS_STR','MAX_BID','MARKET_$','P_REB_M','P_DD_M',
              'P_BLK_M','EXP_GP']].round(2).to_string(index=False))

# ── Excel export ──
OUT = 'fantasy_draft_2026_27_v3_strategy.xlsx'
with pd.ExcelWriter(OUT, engine='openpyxl') as xw:
    board_out = P[P['MARKET_$'] > 0].sort_values('MY_$', ascending=False)
    keep = ['PLAYER_NAME','TEAM','POS_STR','AGE','HEIGHT_IN','EXP_GP','AVAIL','RISK',
            'MY_$','MARKET_$','SURPLUS','MAX_BID'] + ['P_'+c+'_M' for c in RAW]
    board_out[keep].round(3).to_excel(xw, 'Draft Board', index=False)
    my_team[[c for c in ROSTER_COLS if c in my_team.columns]].round(3) \
        .to_excel(xw, 'My Team', index=False)
    df_bundles.drop(columns=['tset']).head(80).round(4) \
        .to_excel(xw, 'Bundle Search', index=False)
    df_robust.to_excel(xw, 'Robustness', index=False)
    summary.to_excel(xw, 'Season Projection', index=False)
    df_sales.to_excel(xw, 'Auction Log', index=False)
    for nm, t in auction_teams.items():
        sheet = ''.join(ch for ch in nm.replace('>>> ', '') if ch.isalnum() or ch == ' ')[:28]
        t[['SLOT','ROLE','PLAYER_NAME','TEAM','POS_STR','PAID','MARKET_$','EXP_GP']] \
            .to_excel(xw, f'T_{sheet}'[:31], index=False)
    df_conc.round(3).to_excel(xw, 'Category Structure', index=False)

print(f'\n✅ exported → {OUT}')
print(f"   sheets: Draft Board · My Team · Bundle Search · Robustness · Season Projection"
      f" · Auction Log · {len(auction_teams)} team sheets · Category Structure")
print('\n' + '='*112)
print('  BOTTOM LINE')
print('='*112)
print(f"  Target {' + '.join(LBL[c] for c in MY_PLAN)} and concede "
      f"{' / '.join(LBL[c] for c in MY_PUNTS)} completely.")
print(f"  Expected: {me.W:.1f}-{me.L:.1f} · {me['PLAYOFF%']:.0f}% playoff odds · "
      f"{me['CATS/wk']:.1f} of 10 categories per matchup.")
print(f"  Hard rule: never bid above MAX_BID, and never spend a dollar in a punt category.")

---
## Cell 17 · Feed the live draft assistant (`draft_app/`)

The Streamlit app in `draft_app/` is what you actually sit in front of on draft night: you log
each pick as ESPN announces it, and it recomputes live inflation, your remaining budget and
slots, and a max bid per remaining player.

It reads two files — `player_pool.csv` and `league_meta.json` — which were previously generated
by the **v2** notebook. That matters for two reasons:

1. **It carried the v2 strategy.** Its punt board was the guard/shooting plan, which Cell 12
   measures at P(win) ≈ 0.56 against ≈ 0.79 for the plan this notebook derives. The app was
   giving correct arithmetic on the wrong target set.
2. **It carried retired players.** Its pool was built before the draftability filter in Cell 6,
   so Russell Westbrook, Kevin Love, Jonas Valanciunas and Lonzo Ball were all still priced and
   recommendable.

This cell rewrites both files from the v3 model. The column names the app already expects are
preserved, so it keeps working without changes — `EST_$PUNT` now carries **your v3 valuation**
and `EST_$BAL` the market price. The v3-specific columns (`MY_$`, `MARKET_$`, `SURPLUS`,
`MAX_BID`) and the target/punt plan are added alongside, and `league_meta.json` now records which
categories the plan targets so the app can label the board instead of hardcoding it.

Nothing you have logged in the app is touched — `draft_log.json` is left alone.

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# CELL 17 — EXPORT FOR THE LIVE DRAFT ASSISTANT (draft_app/)
# ═══════════════════════════════════════════════════════════════════
import datetime, json as _json

APP_DIR = 'draft_app'
if not os.path.isdir(APP_DIR):
    print(f'ℹ  {APP_DIR}/ not found — skipping app export')
else:
    app = P.copy()

    # Column names the existing app already reads, mapped from the v3 model.
    app['POS_ELIGIBLE_STR']    = app['POS_STR']
    app['POS_PRIMARY']         = [sorted(s, key=ALL_POSITIONS.index)[0] for s in app['POS_ELIGIBLE']]
    app['CURRENT_TEAM_ABBR']   = app['TEAM']
    app['TEAMMATE_STRENGTH']   = app['MATES'].round(3)
    app['TEAM_CONTEXT_FACTOR'] = app['CTX'].round(3)
    app['RISK_FLAG']           = app['RISK']
    app['AVAILABILITY']        = app['AVAIL'].round(3)
    app['HEIGHT_TIER']         = pd.cut(app['HEIGHT_IN'], [0, 74, 77, 80, 99],
                                        labels=['small','wing','big wing','big']).astype(str)
    rookie_ids = set(df_meta.loc[
        pd.to_numeric(df_meta['DRAFT_YEAR'], errors='coerce') >= DRAFT_DATE.year, 'PLAYER_ID'])
    app['IS_ROOKIE'] = app['PLAYER_ID'].isin(rookie_ids)

    for cat in COUNTING_CATS:                       # PRED_PTS_PG, PRED_REB_PG, ...
        app[f'PRED_{cat}_PG' if cat != 'FG3M' else 'PRED_FG3M_PG'] = app['P_'+cat+'_M'].round(3)
    app['PRED_FG_PCT']  = np.where(app['P_FGA_M'] > 0, app['P_FGM_M']/app['P_FGA_M'], np.nan).round(4)
    app['PRED_FT_PCT']  = np.where(app['P_FTA_M'] > 0, app['P_FTM_M']/app['P_FTA_M'], np.nan).round(4)
    app['PRED_PTS_MIN'] = np.where(app['P_MIN_M'] > 0, app['P_PTS_M']/app['P_MIN_M'], np.nan).round(4)

    # The app's two boards. EST_$PUNT is now the v3 plan; EST_$BAL stays the market price.
    app['DRAFT_VALUE_TOTAL'] = app['VAL_MARKET'].round(4)
    app['PUNT_VALUE']        = app['VAL_MINE'].round(4)
    app['EST_$BAL']          = app['MARKET_$']
    app['EST_$PUNT']         = app['MY_$']
    app['VAR_BAL']           = app['MARKET_$']      # kept for column compatibility
    app['VAR_PUNT']          = app['MY_$']

    APP_COLS = ['PLAYER_ID','PLAYER_NAME','POS_ELIGIBLE_STR','POS_PRIMARY','AGE','HEIGHT_IN',
                'HEIGHT_TIER','IS_ROOKIE','CURRENT_TEAM_ABBR','TEAM_SOURCE','TEAMMATE_STRENGTH',
                'TEAM_CONTEXT_FACTOR','RISK_FLAG','EXP_GP','AVAILABILITY',
                'PRED_PTS_PG','PRED_AST_PG','PRED_REB_PG','PRED_STL_PG','PRED_BLK_PG',
                'PRED_FG3M_PG','PRED_FG_PCT','PRED_FT_PCT','PRED_DD_PG','PRED_PTS_MIN',
                'DRAFT_VALUE_TOTAL','PUNT_VALUE','VAR_BAL','VAR_PUNT','EST_$BAL','EST_$PUNT',
                # v3 additions
                'MARKET_$','MY_$','SURPLUS','MAX_BID','ON_ROSTER_2627',
                'MODEL_$','MARKET_SOURCE','POS_SOURCE','ESPN_$_RAW','ESPN_$_LIST','ESPN_INJURY',
                'ESPN_PCT_OWNED']
    out = app[APP_COLS].sort_values('MY_$', ascending=False)
    out.to_csv(f'{APP_DIR}/player_pool.csv', index=False)

    meta = {
        'league_name': LEAGUE_NAME, 'your_team_name': YOUR_TEAM_NAME,
        'team_names': TEAM_NAMES, 'num_teams': NUM_TEAMS, 'scoring_type': SCORING_TYPE,
        'category_map': CATEGORY_MAP,
        'draft_type': DRAFT_TYPE, 'draft_date': DRAFT_DATE.isoformat(),
        'budget_per_team': BUDGET_PER_TEAM, 'min_bid': MIN_BID,
        'roster_size': ROSTER_SIZE, 'total_starters': TOTAL_STARTERS,
        'bench_count': BENCH_COUNT, 'ir_slots': IR_SLOTS, 'total_drafted': TOTAL_DRAFTED,
        'max_centers': MAX_CENTERS, 'position_maximums': POSITION_MAXIMUMS,
        'starter_slots': [[n, list(a)] for n, a in STARTER_SLOTS],
        'lineup_changes': LINEUP_CHANGES, 'lineup_protection': LINEUP_PROTECTION,
        'fab_budget': FAB_BUDGET, 'matchup_acquisition_limit': MATCHUP_ACQUISITION_LIMIT,
        'keepers': KEEPERS, 'regular_season_matchups': REGULAR_SEASON_MATCHUPS,
        'weeks_per_matchup': WEEKS_PER_MATCHUP, 'playoff_teams': PLAYOFF_TEAMS,
        'category_tie_breaker': CATEGORY_TIE_BREAKER,
        'durability_exponent': DURABILITY_EXPONENT,
        'cats_to_win': CATS_TO_WIN,
        'surplus_capture': SURPLUS_CAPTURE,   # the app's bid ceilings use this
        # ── what the v3 model decided: the app labels its board from this ──
        'model_version': 'v3-category-dominance',
        'target_cats': [LBL[c] for c in MY_PLAN],
        'punt_cats':   [LBL[c] for c in MY_PUNTS],
        'strategy_label': 'Target ' + '+'.join(LBL[c] for c in MY_PLAN),
        'expected_record': f"{me.W:.1f}-{me.L:.1f}",
        'playoff_odds': float(me['PLAYOFF%']),
        'generated_at': datetime.datetime.now().isoformat(timespec='seconds'),
    }
    with open(f'{APP_DIR}/league_meta.json', 'w') as f:
        _json.dump(meta, f, indent=2, ensure_ascii=False)

    print(f'✅ wrote {APP_DIR}/player_pool.csv  ({len(out)} draftable players, '
          f'{len(APP_COLS)} columns)')
    print(f'✅ wrote {APP_DIR}/league_meta.json (model {meta["model_version"]})')
    print(f'   TARGET: {" ".join(meta["target_cats"])}   PUNT: {" ".join(meta["punt_cats"])}')
    print(f'   every player in the pool is on a 2026-27 NBA roster '
          f'({int(out["ON_ROSTER_2627"].all())==1})')
    print(f'   MARKET_$ source: {out["MARKET_SOURCE"].value_counts().to_dict()}'
          f'  ·  positions: {out["POS_SOURCE"].value_counts().to_dict()}')
    print(f'\n   Run it with:  streamlit run {APP_DIR}/app.py')